# Giai đoạn 19: Phân tích độ nhạy kênh scalp trên ScaleOLDNEW (hậu nghiệm, Addendum A)

Execution log của Phase 17 cho thấy số kênh scalp được giữ **nhiều hơn** số điện cực trong Table 1 ở P01, P02, P03, P05. Những kênh thừa có thể là vị trí không gắn điện cực, và chúng đi vào cả phép CAR.

Phân tích này đã được định nghĩa **trước khi chạy**, trong *Addendum A* của protocol:
- kênh scalp = kênh đã giữ theo v1.6 **∩** danh sách kênh trong file EEGLAB `SEEGxx.set` của tác giả gốc (metadata kiểm chứng được, không cắt cho khớp số đếm);
- tính lại CAR trên các kênh này;
- mọi thứ khác giữ nguyên: cửa sổ, chia calib/test, target, encoder, mô hình, 5 fold.

Kết quả đã đăng ký (v1.6) vẫn là kết quả chính. Phân tích này chỉ kiểm tra xem H1/H2 có phụ thuộc vào các kênh nghi vấn hay không.

| Phần | Nội dung | Runtime |
|---|---|---|
| 19A | So sánh kênh cho 12 bệnh nhân + dựng `primary_setch` | CPU, vài phút |
| 19B | Readout Final/Base, Learned-3, Fixed-3 | GPU, khoảng 2 giờ |
| 19C | Báo cáo H1/H2 hậu nghiệm cạnh kết quả đã đăng ký | vài giây |
| 19D | (tuỳ chọn) đo độ lệch thời gian do resample cho từng session ds004752 | CPU, khoảng 10 phút |

In [ ]:
#@title 1. Mount Drive + đường dẫn
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob
ROOT     = '/content/drive/MyDrive/iEEG_S2I'   #@param {type:"string"}
CODE_DIR = f'{ROOT}/code'
SC_ROOT  = f'{ROOT}/data/scalesoldnew'
DATA17   = f'{ROOT}/data/scales_s2i'
RUNS17   = f'{ROOT}/runs/p17'
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
os.environ['PYTHONPATH'] = CODE_DIR
os.chdir(CODE_DIR)
!pip -q install mne==1.13.2
print('file .set của tác giả:', len(glob.glob(f'{SC_ROOT}/raw/SEEG*/SEEG[0-9]*.set')), '(cần 12, không tính *Intracranial.set)')
print('dữ liệu primary:', len(glob.glob(f'{DATA17}/primary/sub-P[0-9][0-9].pt')), '/ 12')

## 2. Ghi code lên Drive

In [ ]:
%%writefile {CODE_DIR}/s2i/scales17.py
"""
Phase 17 - external validation on ScaleOLDNEW, exactly as in paper/external_validation_protocol.md (v1.6, locked).

Commands
  build   Phase-14 output (proc/<patient>_Block<k>.npz: notch 50/100 Hz, 0.5-70 Hz, 200 Hz, microvolts) + the raw
          ADES files and spreadsheets on Drive -> one data folder per evaluation level, in the s2i format read by
          s2i.readout --strict and s2i.review13 volts:
            primary      all windows, no automatic artefact rejection, no target clipping
            secondary    windows with |z| > 15 (CAR scalp EEG) removed, targets clipped at |z| = 10, median / MAD of
                         the whole block
            sensitivity  the same thresholds, median / MAD of the first 20 % of the windows
          plus the execution log (p17_build.md / p17_build.json): channel rules and the Table-1 consistency check,
          non-finite counts (raw data), recording gaps, windows, split, inclusion, target validity, markers.
  runs    copy each final encoder (s2i.shared --final -> <variant>/enc_final.pt) to one file per patient
          (<runs>/<variant>/enc_sub-Pxx.pt), so that s2i.readout runs on ScaleOLDNEW unchanged
  report  H1 / H2 (paired two-sided Wilcoxon, Holm over H1-H2 only, median direction, patient bootstrap CI of the
          mean), descriptive analyses, constant-prediction counts, common mode by recording setup
          -> p17_report.md / p17_report.json

Rules implemented here (protocol sections in brackets):
  scalp channel [4]   EEG type in the header; name (after the dataset's lookup table) is an MNE 10-05 position; not
                      A1/A2/M1/M2; not bad in any block; non-zero SD in every block; no non-finite raw sample
  target [4, 7]       pair of the BipolarChannels sheet; both contacts recorded in every block, never bad, no
                      non-finite raw sample; bipolar = A - B (microvolts); non-zero variance in calibration and test
  windows [5]         8 s, consecutive, inside each block (file) only, 3.4 s from the block edges; a window
                      overlapping a recording gap (>= 50 ms where all kept scalp channels, or all target contacts, are
                      exactly 0 or non-finite in the RAW data) or its 3.4 s margins is dropped; the last piece of a
                      block is kept if >= 2 s
  split [5]           windows ordered by block and time; n_cal = max(3, round(0.2 N)); 1-window gap; rest = test;
                      patient included only if n_cal >= 5, >= 3 scalp channels and >= 3 valid targets
  CAR [4]             mean of the kept scalp channels subtracted at every sample (microvolts), before normalisation
Storage: every signal is robust-normalised with the median / MAD of ALL kept samples of the patient and stored as
float16 together with these statistics (norm_stats, one pseudo-session "all"); s2i.readout --strict undoes this and
re-normalises with the calibration windows only, exactly as for ds004752.

Usage
-----
python -m s2i.scales17 build  --sc_root SC_ROOT --out DATA17
python -m s2i.scales17 runs   --enc_root ENC17 --data DATA17/primary --runs RUNS17
python -m s2i.scales17 report --data DATA17 --runs RUNS17
"""
import argparse
import glob
import json
import os
import re
import shutil
import sys
import time

import numpy as np

FS = 200
WIN, MIN_REM, EDGE = 8 * FS, 2 * FS, 680              # 8 s windows, >= 2 s remainder, 3.4 s margins (samples)
GAP_SEC = 0.05
CAL_FRAC, MIN_CAL, MIN_SCALP, MIN_TGT = 0.2, 5, 3, 3
REJECT_Z, CLIP_Z = 15.0, 10.0
LEVELS = ("primary", "secondary", "sensitivity")
LOOKUP = {"T3": "T7", "T4": "T8", "T5": "P7", "T6": "P8", "FB1": "FT9", "FB2": "FT10", "TP1": "TP9", "TP2": "TP10"}
EXCLUDED_NAMES = {"A1", "A2", "M1", "M2"}
# Table 1 of Barborica et al. 2023: (patient number, ID, recording setup, number of scalp electrodes).
# Setup from the Methods: patients 1-8 two 128-channel amplifiers, patients 9-12 one 256-channel amplifier.
# Scalp counts transcribed from the article's HTML table; check against the PDF.
PATIENTS = [(1, 89, "two amplifiers", 30), (2, 90, "two amplifiers", 30), (3, 92, "two amplifiers", 30),
            (4, 96, "two amplifiers", 38), (5, 97, "two amplifiers", 35), (6, 98, "two amplifiers", 38),
            (7, 99, "two amplifiers", 38), (8, 101, "two amplifiers", 40), (9, 102, "one amplifier", 40),
            (10, 104, "one amplifier", 40), (11, 105, "one amplifier", 37), (12, 107, "one amplifier", 40)]
CONTACT = re.compile(r"^[A-Z]{1,2}'?\d{1,2}$")


def log(msg):
    print(f"{time.strftime('%H:%M:%S')} | {msg}", flush=True)


def n_cal_of(n):
    return max(3, int(round(n * CAL_FRAC)))


def robust(x):
    med = np.median(x, axis=1, keepdims=True)
    mad = np.median(np.abs(x - med), axis=1, keepdims=True) * 1.4826
    return med, np.where(mad < 1e-30, 1.0, mad)


# ----------------------------------------------------------------------------- inputs
def template_1005():
    import warnings
    import mne
    with warnings.catch_warnings():                   # renamed colin27_1005 from MNE 1.14 (same positions)
        warnings.simplefilter("ignore", FutureWarning)
        try:
            mont = mne.channels.make_standard_montage("standard_1005")
        except ValueError:
            mont = mne.channels.make_standard_montage("colin27_1005")
    return {n.upper(): n for n in mont.ch_names}


def scalp_name(name, t1005):
    """-> (10-05 name or None, reason)."""
    up = LOOKUP.get(name.strip().upper(), name.strip().upper()).upper()
    if up in EXCLUDED_NAMES:
        return None, "A1/A2/M1/M2"
    if up not in t1005:
        return None, "not a 10-05 position"
    return t1005[up], ""


def load_block(path):
    z = np.load(path, allow_pickle=True)
    out = {k: z[k] for k in ("eeg", "seeg")}
    for k in ("eeg_names", "seeg_names", "bad"):
        out[k] = [str(x).strip() for x in z[k]]
    out["markers"] = [(str(m[0]).strip(), float(m[1])) for m in z["markers"]]
    out["fs_raw"], out["n_raw"] = float(z["fs_raw"]), int(z["n_raw"])
    return out


def read_header(path):
    fs, n, chans = None, None, []
    for ln in open(path, encoding="latin-1").read().splitlines():
        if ln.startswith("#") or "=" not in ln:
            continue
        k, v = [x.strip() for x in ln.split("=", 1)]
        if k == "samplingRate":
            fs = float(v)
        elif k == "numberOfSamples":
            n = int(v)
        else:
            chans.append((k, v))
    return fs, n, chans


def raw_scan(dat, ades, count_names, scalp, contacts, chunk=32768):
    """
    One pass over a raw block (float32, sample-major, microvolts): non-finite samples per channel of count_names,
    and recording gaps = runs of >= 50 ms in which all `scalp` channels, or all `contacts`, are exactly zero or
    non-finite. Gaps are returned as [start, stop) intervals at 200 Hz.
    """
    fs, n, chans = read_header(ades)
    col = {}
    for i, (c, _) in enumerate(chans):
        col.setdefault(c, i)
    assert os.path.getsize(dat) == 4 * len(chans) * n, f"size mismatch {dat}"
    X = np.memmap(dat, dtype="<f4", mode="r", shape=(n, len(chans)))
    cc = [col[c] for c in count_names]
    pos = {c: j for j, c in enumerate(count_names)}
    si, ti = [pos[c] for c in scalp], [pos[c] for c in contacts]
    nonfin = np.zeros(len(cc), np.int64)
    dead = np.zeros(n, bool)
    for k in range(0, n, chunk):
        x = np.asarray(X[k:k + chunk])[:, cc]
        bad = ~np.isfinite(x)
        nonfin += bad.sum(0)
        z = bad | (x == 0)
        dead[k:k + chunk] = (z[:, si].all(1) if si else False) | (z[:, ti].all(1) if ti else False)
    del X
    gaps = []
    d = np.diff(np.r_[0, dead.astype(np.int8), 0])
    for s0, s1 in zip(np.where(d == 1)[0], np.where(d == -1)[0]):
        if s1 - s0 >= int(np.ceil(GAP_SEC * fs)):
            gaps.append((int(np.floor(s0 * FS / fs)), int(np.ceil(s1 * FS / fs))))
    return dict(zip(count_names, nonfin.tolist())), gaps, fs


def windows_of(L, gaps):
    """Consecutive 8-s windows of one block -> (kept [(s0, s1)], number dropped for gaps, remainder kept / dropped)."""
    blocked = np.zeros(L, bool)
    blocked[:EDGE] = blocked[L - EDGE:] = True
    for g0, g1 in gaps:
        blocked[max(0, g0 - EDGE):min(L, g1 + EDGE)] = True
    kept, n_gap, rem = [], 0, "none"
    s, end = EDGE, L - EDGE
    while s < end:
        e = min(s + WIN, end)
        if e - s < WIN:
            if e - s < MIN_REM:
                rem = f"dropped ({(e - s) / FS:.1f} s)"
                break
            rem = f"kept ({(e - s) / FS:.1f} s)"
        if blocked[s:e].any():
            n_gap += 1
        else:
            kept.append((s, e))
        s = e
    return kept, n_gap, rem


# ----------------------------------------------------------------------------- one patient
def build_patient(a, num, pid, setup, t1005_n, t1005):
    sid = f"sub-P{num:02d}"
    raw_dir = os.path.join(a.sc_root, "raw", f"SEEG{pid}")
    files = sorted(glob.glob(os.path.join(a.sc_root, "proc", f"SEEG{pid}_Block*.npz")),
                   key=lambda p: int(re.search(r"Block(\d+)", p).group(1)))
    info = {"subject": sid, "patient": num, "id": pid, "setup": setup, "table1_scalp": t1005_n, "excluded": None}
    if not files:
        info["excluded"] = "no preprocessed block"
        return info, None
    blocks, nums = [], []
    for f in files:
        try:
            blocks.append(load_block(f))
            nums.append(int(re.search(r"Block(\d+)", f).group(1)))
        except Exception as e:                                         # unreadable block: skipped and reported
            info.setdefault("unreadable_blocks", []).append(f"{os.path.basename(f)}: {type(e).__name__}")
    info["blocks"] = nums
    bad_all = sorted({c for b in blocks for c in b["bad"]})
    info["bad_any_block"] = bad_all

    # ---- scalp channels (rules 1-5; non-finite from the raw scan below)
    cand = [c for c in blocks[0]["eeg_names"] if all(c in b["eeg_names"] for b in blocks)]
    reasons, keep, canon = {}, [], []
    for c in cand:
        nm, why = scalp_name(c, t1005)
        if nm is None:
            reasons[c] = why
        elif c in bad_all:
            reasons[c] = "bad"
        elif nm in canon:
            reasons[c] = f"duplicate of {nm}"
        elif any(np.std(b["eeg"][b["eeg_names"].index(c)].astype(np.float64)) == 0 for b in blocks):
            reasons[c] = "zero SD in a block"
        else:
            keep.append(c)
            canon.append(nm)

    # ---- targets: BipolarChannels pairs, contacts recorded in every block and never bad
    import pandas as pd
    xl = sorted(glob.glob(os.path.join(raw_dir, "*ChannelData.xlsx")))
    sheet = pd.read_excel(xl[0], sheet_name="BipolarChannels")
    recorded = set.intersection(*[set(b["seeg_names"]) for b in blocks])
    pairs, t_reason = [], {}
    for _, row in sheet.iterrows():
        nm = str(row["name"]).strip()
        parts = nm.split("-")
        if len(parts) != 2:
            t_reason[nm] = "name not A-B"
            continue
        p0, p1 = parts[0].strip(), parts[1].strip()
        if p0 not in recorded or p1 not in recorded:
            t_reason[nm] = "contact not recorded in every block"
        elif p0 in bad_all or p1 in bad_all:
            t_reason[nm] = "contact bad"
        else:
            pairs.append({"name": nm, "a": p0, "b": p1, "mni": [float(row[k]) for k in ("x_mni", "y_mni", "z_mni")],
                          "structure": str(row.get("structure", ""))})

    # ---- raw scan: non-finite counts and recording gaps (repeated if the non-finite rule changes the sets)
    seeg_all = [c for c in blocks[0]["seeg_names"] if c in recorded]
    count_names = cand + seeg_all
    nonfin, gaps, fs_raw = {c: 0 for c in count_names}, {}, {}
    for it in range(3):
        contacts = sorted({p["a"] for p in pairs} | {p["b"] for p in pairs})
        nonfin = {c: 0 for c in count_names}
        for k, b in zip(nums, blocks):
            base = os.path.join(raw_dir, f"Block{k}", f"SEEG{pid}Block{k}")
            nf, gaps[k], fs_raw[k] = raw_scan(base + ".dat", base + ".ades", count_names, keep, contacts)
            for c, v in nf.items():
                nonfin[c] += v
        bad_scalp = [c for c in keep if nonfin[c] > 0]
        bad_ct = {c for c in contacts if nonfin[c] > 0}
        if not bad_scalp and not bad_ct:
            break
        for c in bad_scalp:
            reasons[c] = "non-finite raw sample"
            canon.pop(keep.index(c))
            keep.remove(c)
        for p in [p for p in pairs if p["a"] in bad_ct or p["b"] in bad_ct]:
            t_reason[p["name"]] = "contact non-finite"
            pairs.remove(p)
    info.update(scalp_candidates=len(cand), scalp_kept=canon, n_scalp=len(keep), scalp_excluded=reasons,
                nonfinite={c: v for c, v in nonfin.items() if v > 0}, nonfinite_channels_checked=len(count_names),
                n_pairs_sheet=int(len(sheet)), pairs_excluded=t_reason, fs_raw=fs_raw,
                gaps={k: v for k, v in gaps.items()})
    contacts = sorted({p["a"] for p in pairs} | {p["b"] for p in pairs})

    # ---- signals per block (microvolts) and windows
    S, W, wl = {}, [], {}
    for k, b in zip(nums, blocks):
        e = np.stack([b["eeg"][b["eeg_names"].index(c)] for c in keep]).astype(np.float64)
        m = np.stack([b["seeg"][b["seeg_names"].index(c)] for c in contacts]).astype(np.float64) if contacts \
            else np.zeros((0, e.shape[1]))
        L = min(e.shape[1], m.shape[1])
        e, m = e[:, :L], m[:, :L]
        ci = {c: i for i, c in enumerate(contacts)}
        bip = np.stack([m[ci[p["a"]]] - m[ci[p["b"]]] for p in pairs]) if pairs else np.zeros((0, L))
        S[k] = {"orig": e, "car": e - e.mean(0, keepdims=True), "mono": m, "bip": bip}
        kw, n_gap, rem = windows_of(L, gaps[k])
        stim = [(c, t) for c, t in b["markers"] if c.isdigit()]
        codes = {}
        for c, _ in stim:
            codes[c] = codes.get(c, 0) + 1
        wl[k] = {"minutes": L / FS / 60, "windows": len(kw), "dropped_gap": n_gap, "remainder": rem,
                 "gaps": len(gaps[k]), "gap_seconds": sum(g1 - g0 for g0, g1 in gaps[k]) / FS, "stim_markers": codes,
                 "other_markers": sorted({c for c, _ in b["markers"] if not c.isdigit()})}
        onsets = np.array([int(round(t * FS)) for _, t in stim], int)
        for s0, s1 in kw:
            W.append({"block": k, "s0": s0, "s1": s1, "onsets": (onsets[(onsets >= s0) & (onsets < s1)] - s0).tolist()})
    info["block_windows"] = wl
    info["n_windows"] = len(W)
    if len(keep) < MIN_SCALP:
        info["excluded"] = f"{len(keep)} scalp channels (< {MIN_SCALP})"
    meta = dict(sid=sid, pid=pid, setup=setup, canon=canon, pairs=pairs, contacts=contacts)
    return info, (S, W, meta)


def level_windows(level, S, W):
    """-> kept windows and their signals for one evaluation level (rejection / clipping as in the protocol)."""
    sig = [{k: S[w["block"]][k][:, w["s0"]:w["s1"]] for k in ("orig", "car", "mono", "bip")} for w in W]
    if level == "primary":
        return W, sig, {}
    if level == "secondary":                          # whole-block median / MAD
        st = {k: {b: robust(S[b][k]) for b in S} for k in ("car", "bip")}
        get = lambda k, w: st[k][w["block"]]                                    # noqa: E731
    else:                                             # first 20 % of the windows, fixed for all later windows
        nc = n_cal_of(len(W))
        st = {k: robust(np.concatenate([s[k] for s in sig[:nc]], 1)) for k in ("car", "bip")}
        get = lambda k, w: st[k]                                                # noqa: E731
    kw, ks = [], []
    for w, s in zip(W, sig):
        med, mad = get("car", w)
        if np.abs((s["car"] - med) / mad).max() > REJECT_Z:
            continue
        s = dict(s)                                   # monopolar signals are used at the primary level only
        med, mad = get("bip", w)
        s["bip"] = np.clip(s["bip"], med - CLIP_Z * mad, med + CLIP_Z * mad)
        kw.append(w)
        ks.append(s)
    return kw, ks, {"rejected_windows": len(W) - len(kw)}


def write_level(a, level, info, data):
    import torch
    S, W, meta = data
    kw, ks, extra = level_windows(level, S, W)
    N = len(kw)
    nc = n_cal_of(N)
    out = {"n_windows": N, "n_cal": nc, "n_test": max(N - nc - 1, 0), **extra,
           "calib_minutes": sum(w["s1"] - w["s0"] for w in kw[:nc]) / FS / 60,
           "test_minutes": sum(w["s1"] - w["s0"] for w in kw[nc + 1:]) / FS / 60, "excluded": info["excluded"]}
    if out["excluded"] is None and nc < MIN_CAL:
        out["excluded"] = f"n_cal = {nc} (< {MIN_CAL}; N = {N})"
    valid, why = [], {}
    for j, p in enumerate(meta["pairs"]):
        vc = np.var(np.concatenate([s["bip"][j] for s in ks[:nc]])) if nc and ks else 0.0
        vt = np.var(np.concatenate([s["bip"][j] for s in ks[nc + 1:]])) if N > nc + 1 else 0.0
        if vc > 0 and vt > 0:
            valid.append(j)
        else:
            why[p["name"]] = "zero variance in calibration or test windows"
    out["n_valid_targets"], out["targets_zero_variance"] = len(valid), why
    if out["excluded"] is None and len(valid) < MIN_TGT:
        out["excluded"] = f"{len(valid)} valid targets (< {MIN_TGT})"
    if out["excluded"] is not None:
        return out
    pairs = [meta["pairs"][j] for j in valid]
    cts = sorted({p["a"] for p in pairs} | {p["b"] for p in pairs})
    ci = [meta["contacts"].index(c) for c in cts]
    arrays = {"eeg_orig": np.concatenate([s["orig"] for s in ks], 1),
              "eeg_car": np.concatenate([s["car"] for s in ks], 1),
              "ieeg_bip": np.concatenate([s["bip"][valid] for s in ks], 1)}
    if level == "primary":                            # monopolar only for the common-mode analysis (Section 8)
        arrays["ieeg_mono"] = np.concatenate([s["mono"][ci] for s in ks], 1)
    d = {"subject": meta["sid"], "dataset": "ScaleOLDNEW", "id": meta["pid"], "setup": meta["setup"], "fs": FS,
         "level": level, "eeg_names": meta["canon"], "contact_names": cts,
         "bip_names": [p["name"] for p in pairs], "bip_structure": [p["structure"] for p in pairs],
         "bip_mni": torch.tensor([p["mni"] for p in pairs], dtype=torch.float32),
         "bip_pairs": [(cts.index(p["a"]), cts.index(p["b"])) for p in pairs],
         "window_block": [w["block"] for w in kw], "window_start": [w["s0"] for w in kw],
         "stim_events": [(i, o) for i, w in enumerate(kw) for o in w["onsets"]],
         "norm_clip_z": float("inf") if level == "primary" else CLIP_Z, "norm_stats": {}}
    bounds, c = [], 0
    for w in kw:
        bounds.append((c, c + w["s1"] - w["s0"]))
        c += w["s1"] - w["s0"]
    d["trial_bounds"] = torch.tensor(bounds, dtype=torch.long)
    d["trial_session"] = ["all"] * N
    n_inf = 0
    for k, x in arrays.items():                       # storage normalisation (undone by readout --strict)
        med, mad = robust(x)
        z = (x - med) / mad
        n_inf += int((np.abs(z) > 6e4).sum())
        d[k] = torch.from_numpy(np.clip(z, -6e4, 6e4).astype(np.float16))
        d["norm_stats"][k] = {"all": {"med": med[:, 0].tolist(), "mad": mad[:, 0].tolist()}}
    out["float16_overflow_samples"] = n_inf
    od = os.path.join(a.out, level)
    os.makedirs(od, exist_ok=True)
    fp = os.path.join(od, f"{meta['sid']}.pt")
    torch.save(d, fp + ".tmp")
    os.replace(fp + ".tmp", fp)
    torch.save({k: d[k] for k in ("subject", "norm_stats", "norm_clip_z", "trial_bounds", "trial_session")},
               os.path.join(od, f"{meta['sid']}_normstats.pt"))
    out["stim_events_in_windows"] = len(d["stim_events"])
    return out


def filter_record():
    """Filter / resampling configuration of the installed MNE (the Phase-14 preprocessing used the same functions)."""
    import inspect
    import mne
    from s2i.data.preprocess_ds004752 import BAND, NOTCH
    h = mne.filter.create_filter(None, 4096.0, BAND[0], BAND[1], verbose=False)
    sig = lambda f: {k: str(v.default) for k, v in inspect.signature(f).parameters.items()      # noqa: E731
                     if v.default is not inspect.Parameter.empty}
    return {"mne": mne.__version__, "notch_hz": list(NOTCH), "band_hz": list(BAND),
            "bandpass_length_at_4096_hz_s": len(h) / 4096.0,
            "notch_filter_defaults": sig(mne.io.BaseRaw.notch_filter), "filter_defaults": sig(mne.io.BaseRaw.filter),
            "resample_defaults": sig(mne.io.BaseRaw.resample)}


def build(a):
    os.makedirs(a.out, exist_ok=True)
    t1005 = template_1005()
    jp = os.path.join(a.out, "p17_build.json")
    res = json.load(open(jp)) if os.path.exists(jp) else {"patients": {}}
    res["filters"] = filter_record()
    lt = os.path.join(a.sc_root, "raw", "ElectrodeLabelsLookupTable.txt")
    res["lookup_table_file"] = open(lt, encoding="latin-1").read() if os.path.exists(lt) else "not found"
    res["rules"] = {"window_s": WIN / FS, "edge_margin_s": EDGE / FS, "min_remainder_s": MIN_REM / FS,
                    "gap_min_s": GAP_SEC, "cal_frac": CAL_FRAC, "min_cal": MIN_CAL, "min_scalp": MIN_SCALP,
                    "min_targets": MIN_TGT, "reject_z": REJECT_Z, "clip_z": CLIP_Z, "lookup": LOOKUP}
    for num, pid, setup, t1 in PATIENTS:
        sid = f"sub-P{num:02d}"
        if a.patients and sid not in a.patients and str(pid) not in a.patients:
            continue
        if sid in res["patients"] and all(lv in res["patients"][sid].get("levels", {}) for lv in LEVELS):
            log(f"[{sid}] already built")
            continue
        t0 = time.time()
        info, data = build_patient(a, num, pid, setup, t1, t1005)
        info["levels"] = {}
        if data is not None:
            for lv in LEVELS:
                info["levels"][lv] = write_level(a, lv, info, data)
        del data
        res["patients"][sid] = info
        json.dump(res, open(jp + ".tmp", "w"), indent=1, default=str)
        os.replace(jp + ".tmp", jp)
        lv = info["levels"].get("primary", {})
        log(f"[{sid}] ID {pid}: {info.get('n_scalp', 0)} scalp (Table 1: {t1}), "
            f"{lv.get('n_valid_targets', 0)} targets, {lv.get('n_windows', 0)} windows "
            f"(calib {lv.get('n_cal', 0)}, test {lv.get('n_test', 0)}), excluded: {lv.get('excluded', info['excluded'])} | "
            f"{(time.time() - t0) / 60:.1f} min")
    write_build_md(a, res)


def write_build_md(a, res):
    P = res["patients"]
    f = res["filters"]
    L = ["# Phase 17 execution log: data (protocol v1.6, Sections 4, 5, 9)", "",
         f"- MNE {f['mne']}; notch {f['notch_hz']} Hz, band-pass {f['band_hz']} Hz (band-pass FIR length at 4096 Hz: "
         f"{f['bandpass_length_at_4096_hz_s']:.2f} s); defaults: notch {f['notch_filter_defaults']}, "
         f"filter {f['filter_defaults']}, resample {f['resample_defaults']}",
         f"- rules: {res['rules']}", "", "## Scalp channels (consistency check against Table 1; not used to change rules)", "",
         "| patient | ID | setup | blocks | candidates (EEG in every block) | kept | Table 1 | excluded (reason: n) |",
         "|---|---|---|---|---|---|---|---|"]
    for sid, i in P.items():
        rs = {}
        for _, why in i.get("scalp_excluded", {}).items():
            rs[why] = rs.get(why, 0) + 1
        L.append(f"| {sid} | {i['id']} | {i['setup']} | {len(i.get('blocks', []))} | {i.get('scalp_candidates', 0)} | "
                 f"{i.get('n_scalp', 0)} | {i['table1_scalp']} | " + ", ".join(f"{k}: {v}" for k, v in rs.items()) + " |")
    L += ["", "Kept scalp channels per patient:", ""]
    L += [f"- {sid}: {', '.join(i.get('scalp_kept', []))}" for sid, i in P.items()]
    L += ["", "## Non-finite raw samples (all scalp candidates and recorded contacts; channels with 0 not listed)", ""]
    for sid, i in P.items():
        nf = i.get("nonfinite", {})
        L.append(f"- {sid}: {i.get('nonfinite_channels_checked', 0)} channels checked; "
                 + (", ".join(f"{c}: {v}" for c, v in nf.items()) if nf else "0 non-finite samples"))
    L += ["", "## Targets", "", "| patient | pairs in sheet | after recording / bad / non-finite rules | valid (primary) | "
          "excluded (reason: n) |", "|---|---|---|---|---|"]
    for sid, i in P.items():
        rs = {}
        for _, why in i.get("pairs_excluded", {}).items():
            rs[why] = rs.get(why, 0) + 1
        lv = i["levels"].get("primary", {})
        for why in lv.get("targets_zero_variance", {}).values():
            rs[why] = rs.get(why, 0) + 1
        L.append(f"| {sid} | {i.get('n_pairs_sheet', 0)} | {i.get('n_pairs_sheet', 0) - len(i.get('pairs_excluded', {}))} | "
                 f"{lv.get('n_valid_targets', 0)} | " + ", ".join(f"{k}: {v}" for k, v in rs.items()) + " |")
    L += ["", "## Blocks, gaps, windows and markers", "",
          "| patient | block | min | raw Hz | gaps (n, s) | windows kept | dropped (gap) | last piece | stimulus markers | other markers |",
          "|---|---|---|---|---|---|---|---|---|---|"]
    for sid, i in P.items():
        for k, b in i.get("block_windows", {}).items():
            L.append(f"| {sid} | {k} | {b['minutes']:.1f} | {i['fs_raw'].get(k, i['fs_raw'].get(str(k), ''))} | "
                     f"{b['gaps']}, {b['gap_seconds']:.1f} | {b['windows']} | {b['dropped_gap']} | {b['remainder']} | "
                     f"{b['stim_markers']} | {len(b['other_markers'])} |")
    for lv in LEVELS:
        L += ["", f"## Split and inclusion: {lv}", "",
              "| patient | windows | rejected | calib | test | calib min | test min | valid targets | float16 clipped | excluded |",
              "|---|---|---|---|---|---|---|---|---|---|"]
        for sid, i in P.items():
            x = i["levels"].get(lv)
            if x is None:
                L.append(f"| {sid} | | | | | | | | | {i.get('excluded')} |")
                continue
            L.append(f"| {sid} | {x['n_windows']} | {x.get('rejected_windows', 0)} | {x['n_cal']} | {x['n_test']} | "
                     f"{x['calib_minutes']:.2f} | {x['test_minutes']:.1f} | {x['n_valid_targets']} | "
                     f"{x.get('float16_overflow_samples', '')} | {x['excluded'] or ''} |")
        inc = [s for s, i in P.items() if i["levels"].get(lv, {}).get("excluded") is None and i["levels"].get(lv)]
        L.append(f"\nincluded: {len(inc)} patients")
    out = "\n".join(L)
    open(os.path.join(a.out, "p17_build.md"), "w", encoding="utf-8").write(out + "\n")
    print(out)


# ----------------------------------------------------------------------------- runs
def runs(a):
    subs = sorted(os.path.basename(p)[:-3] for p in glob.glob(os.path.join(a.data, "sub-P*.pt"))
                  if not p.endswith("_normstats.pt"))
    for v in a.variants:
        src = os.path.join(a.enc_root, v, "enc_final.pt")
        assert os.path.exists(src), f"missing final encoder {src} (run s2i.shared --final first)"
        od = os.path.join(a.runs, v)
        os.makedirs(od, exist_ok=True)
        for s in subs:
            dst = os.path.join(od, f"enc_{s}.pt")
            if not os.path.exists(dst):
                shutil.copy(src, dst)
        log(f"{v}: encoder copied for {len(subs)} patients -> {od}")


# ----------------------------------------------------------------------------- report
def _load(run, name):
    out = {}
    for f in glob.glob(os.path.join(run, name, "readout_sub-P*.json")):
        d = json.load(open(f))
        out[d["subject"]] = d
    return out


def _ex(D, m, k="r"):
    return {s: r["explicit"][m]["mean"][k] for s, r in D.items() if m in r.get("explicit", {})
            and k in r["explicit"][m]["mean"]}


def compare(a_, b_, rng):
    from scipy.stats import wilcoxon
    s = sorted(set(a_) & set(b_))
    x, y = np.array([a_[q] for q in s]), np.array([b_[q] for q in s])
    d = x - y
    if len(d) == 0:
        return None
    bt = np.array([d[rng.integers(0, len(d), len(d))].mean() for _ in range(10000)])
    try:
        p = float(wilcoxon(x, y, alternative="two-sided").pvalue)
    except ValueError:
        p = float("nan")
    return {"n": len(d), "mean_a": float(x.mean()), "mean_b": float(y.mean()), "median_diff": float(np.median(d)),
            "mean_diff": float(d.mean()), "ci": [float(np.percentile(bt, 2.5)), float(np.percentile(bt, 97.5))],
            "wins": int((d > 0).sum()), "p": p, "per_patient": dict(zip(s, d.round(5).tolist()))}


def report(a):
    rng = np.random.default_rng(0)
    R = a.runs
    B = json.load(open(os.path.join(a.data, "p17_build.json")))["patients"]
    M = {}
    for lv in LEVELS:
        M[lv] = {"final": _load(os.path.join(R, "main"), f"final_{lv}"),
                 "learned3": _load(os.path.join(R, "no_env"), f"learned3_{lv}"),
                 "fixed3": _load(os.path.join(R, "no_env"), f"fixed3_{lv}")}
    fixed1 = _load(os.path.join(R, "no_env"), "fixed1_primary")
    f10 = _load(os.path.join(R, "main"), "final_primary_f10")
    P = M["primary"]
    included = sorted(s for s, i in B.items() if i["levels"].get("primary", {}).get("excluded") is None
                      and i["levels"].get("primary"))
    out, L = {}, ["# Phase 17: external validation on ScaleOLDNEW (protocol v1.6)", ""]
    have = {k: sorted(v) for k, v in P.items()}
    L.append(f"- included patients (primary level): {len(included)}: {', '.join(included)}")
    for k, v in have.items():
        if v != included:
            L.append(f"- **WARNING**: results for {k} exist for {len(v)} patients ({', '.join(v)}): run incomplete")
    # ---- confirmatory
    h1 = compare(_ex(P["final"], "combo_nosess"), _ex(P["final"], "base"), rng)
    h2 = compare(_ex(P["learned3"], "combo_nosess"), _ex(P["fixed3"], "combo_nosess"), rng)
    ps = [h1["p"], h2["p"]]
    order = np.argsort(ps)
    adj = [0.0, 0.0]
    adj[order[0]] = min(1.0, 2 * ps[order[0]])
    adj[order[1]] = max(adj[order[0]], min(1.0, ps[order[1]]))
    L += ["", "## Primary hypotheses (primary level, Pearson r; two-sided Wilcoxon, Holm over H1 and H2)", "",
          "| hypothesis | A | B | n | mean A | mean B | median A − B | mean A − B [95 % CI] | A > B | p | p (Holm) | supported |",
          "|---|---|---|---|---|---|---|---|---|---|---|---|"]
    for name, (la, lb), h, pa in (("H1 (system)", ("Final", "Base"), h1, adj[0]),
                                  ("H2 (learning)", ("Learned-3", "Fixed-3"), h2, adj[1])):
        sup = pa < 0.05 and h["median_diff"] > 0
        h["p_holm"], h["supported"] = pa, bool(sup)
        L.append(f"| {name} | {la} | {lb} | {h['n']} | {h['mean_a']:.4f} | {h['mean_b']:.4f} | {h['median_diff']:+.4f} | "
                 f"{h['mean_diff']:+.4f} [{h['ci'][0]:+.4f}, {h['ci'][1]:+.4f}] | {h['wins']}/{h['n']} | {h['p']:.5f} | "
                 f"{pa:.5f} | {'**yes**' if sup else 'no'} |")
    out["H1"], out["H2"] = h1, h2
    L += ["", "Per-patient differences (A − B):", ""]
    for name, h in (("H1", h1), ("H2", h2)):
        L.append(f"- {name}: " + ", ".join(f"{s} {v:+.4f}" for s, v in h["per_patient"].items()))
    # ---- descriptive
    L += ["", "## Descriptive analyses (no confirmatory claim; unadjusted p)", "",
          "| level | comparison | metric | n | mean A | mean B | mean A − B [95 % CI] | median A − B | A > B | p |",
          "|---|---|---|---|---|---|---|---|---|---|"]
    desc = []

    def row(lv, lab, A, B, metric):
        h = compare(A, B, rng)
        if h is None:
            return
        desc.append({"level": lv, "comparison": lab, "metric": metric, **h})
        L.append(f"| {lv} | {lab} | {metric} | {h['n']} | {h['mean_a']:.4f} | {h['mean_b']:.4f} | {h['mean_diff']:+.4f} "
                 f"[{h['ci'][0]:+.4f}, {h['ci'][1]:+.4f}] | {h['median_diff']:+.4f} | {h['wins']}/{h['n']} | {h['p']:.4f} |")
    row("primary", "Final − Fixed-1", _ex(P["final"], "combo_nosess"), _ex(fixed1, "combo_nosess"), "r")
    for lv in ("secondary", "sensitivity"):
        row(lv, "H1: Final − Base", _ex(M[lv]["final"], "combo_nosess"), _ex(M[lv]["final"], "base"), "r")
        row(lv, "H2: Learned-3 − Fixed-3", _ex(M[lv]["learned3"], "combo_nosess"), _ex(M[lv]["fixed3"], "combo_nosess"), "r")
    for metric in ("r_event", "r_no50", "r2"):
        row("primary", "Final − Base", _ex(P["final"], "combo_nosess", metric), _ex(P["final"], "base", metric), metric)
        row("primary", "Learned-3 − Fixed-3", _ex(P["learned3"], "combo_nosess", metric),
            _ex(P["fixed3"], "combo_nosess", metric), metric)
    mins = lambda D, f: np.mean([r["fracs"][f]["minutes"] for r in D.values()]) if D else float("nan")   # noqa: E731
    row("primary", f"Final, 10 % ({mins(f10, '0.1'):.1f} min) − Base, 20 % ({mins(P['final'], '0.2'):.1f} min)",
        _ex(f10, "combo_nosess"), _ex(P["final"], "base"), "r")
    out["descriptive"] = desc
    L += ["", "Mean over patients, primary level:", "", "| model | r | R² | r_event | r_no50 | constant predictions |",
          "|---|---|---|---|---|---|"]
    for lab, D, m in (("Base", P["final"], "base"), ("Fixed-1", fixed1, "combo_nosess"),
                      ("Fixed-3", P["fixed3"], "combo_nosess"), ("Learned-3", P["learned3"], "combo_nosess"),
                      ("Final", P["final"], "combo_nosess"), ("Final, 10 % calibration", f10, "combo_nosess")):
        v = {k: _ex(D, m, k) for k in ("r", "r2", "r_event", "r_no50")}
        nc = sum(r["explicit"][m].get("n_const", 0) for r in D.values() if m in r.get("explicit", {}))
        L.append(f"| {lab} | " + " | ".join(f"{np.mean(list(x.values())):.4f}" if x else "" for x in v.values())
                 + f" | {nc} |")
    # ---- common mode (Section 8)
    cm = {}
    for f in glob.glob(os.path.join(R, "review13", "volts_sub-P*.json")):
        d = json.load(open(f))
        cm[d["subject"]] = d
    if cm:
        L += ["", "## Reference / common-mode analysis (descriptive; lagged ridge, primary level, volts)", "",
              "| patient | setup | scalp-cm vs iEEG-cm r | mono, orig | mono − cm, orig | bip, orig | bip, CAR | cm, orig | "
              "removal of iEEG cm | interaction |", "|---|---|---|---|---|---|---|---|---|---|"]
        grp = {}
        for s in sorted(cm):
            q, p_ = cm[s], cm[s]["pred"]
            rem = p_["mono|orig"] - p_["mono_minus_cm|orig"]
            inter = p_["mono|orig"] - p_["mono|car"] - p_["bip|orig"] + p_["bip|car"]
            setup = B[s]["setup"]
            grp.setdefault(setup, []).append((q["r_scalp_cm_vs_ieeg_cm"], p_["mono|orig"], rem, inter))
            L.append(f"| {s} | {setup} | {q['r_scalp_cm_vs_ieeg_cm']:+.2f} | {p_['mono|orig']:.3f} | "
                     f"{p_['mono_minus_cm|orig']:.3f} | {p_['bip|orig']:.3f} | {p_['bip|car']:.3f} | {p_['cm|orig']:.3f} | "
                     f"{-rem:+.3f} | {inter:+.3f} |")
        L += ["", "| setup | n | mean scalp-cm vs iEEG-cm r | mean abs r | mean mono, orig | mean removal of iEEG cm | "
              "mean interaction |", "|---|---|---|---|---|---|---|"]
        for g, v in sorted(grp.items()):
            v = np.array(v)
            L.append(f"| {g} | {len(v)} | {v[:, 0].mean():+.3f} | {np.abs(v[:, 0]).mean():.3f} | {v[:, 1].mean():.3f} | "
                     f"{-v[:, 2].mean():+.3f} | {v[:, 3].mean():+.3f} |")
        out["common_mode"] = {s: {"setup": B[s]["setup"], **cm[s]} for s in cm}
    md = "\n".join(L)
    print(md)
    open(os.path.join(R, "p17_report.md"), "w", encoding="utf-8").write(md + "\n")
    json.dump(out, open(os.path.join(R, "p17_report.json"), "w"), indent=1)


# ----------------------------------------------------------------------------- post-hoc channel sensitivity
def set_channels(path):
    """Channel labels of an EEGLAB .set file (the original authors' preprocessed scalp data)."""
    from scipy.io import loadmat
    try:
        m = loadmat(path, squeeze_me=True, struct_as_record=False)
        E = m["EEG"] if "EEG" in m else None
        locs = E.chanlocs if E is not None else m["chanlocs"]
        return [str(c.labels).strip() for c in np.atleast_1d(locs)]
    except NotImplementedError:                         # MATLAB v7.3 (HDF5)
        import h5py
        with h5py.File(path, "r") as f:
            g = f["EEG"]["chanlocs"]["labels"] if "EEG" in f else f["chanlocs"]["labels"]
            return ["".join(chr(int(c)) for c in np.ravel(f[r[0]][()])) for r in g]


def setch(a):
    """
    Post-hoc sensitivity analysis (protocol v1.6, Addendum A). Scalp channels = channels kept under v1.6 that also
    appear in the original authors' EEGLAB scalp file (raw/SEEGxx/SEEGxx.set), i.e. verified electrodes. CAR is
    recomputed on these channels; targets, windows, split and everything else are those of the primary level.
    -> <out>/primary_setch/sub-Pxx.pt (+ _normstats.pt) and <out>/p17_setch.md (channel comparison table).
    """
    import torch
    t1005 = template_1005()
    src = os.path.join(a.out, "primary")
    dst = os.path.join(a.out, "primary_setch")
    os.makedirs(dst, exist_ok=True)
    L = ["# Post-hoc channel sensitivity (Addendum A): scalp channels confirmed by the original authors' EEGLAB file", "",
         "| patient | ID | Table 1 | kept (v1.6) | authors' .set channels (10-05 names) | kept ∩ .set | kept, not in .set | "
         ".set, not kept | status |", "|---|---|---|---|---|---|---|---|---|"]
    info = {}
    for num, pid, setup, t1 in PATIENTS:
        sid = f"sub-P{num:02d}"
        fp = os.path.join(src, f"{sid}.pt")
        sp = os.path.join(a.sc_root, "raw", f"SEEG{pid}", f"SEEG{pid}.set")
        if not os.path.exists(fp):
            L.append(f"| {sid} | {pid} | {t1} | – | – | – | – | – | not in primary |")
            continue
        if not os.path.exists(sp):
            L.append(f"| {sid} | {pid} | {t1} | – | – | – | – | – | no .set file |")
            continue
        d = torch.load(fp, weights_only=False)
        kept = list(d["eeg_names"])
        raw_set = set_channels(sp)
        canon_set = []
        for c in raw_set:
            nm, _ = scalp_name(c, t1005)
            if nm is not None and nm not in canon_set:
                canon_set.append(nm)
        keep = [c for c in kept if c in canon_set]
        status = "ok" if len(keep) >= MIN_SCALP else f"excluded (< {MIN_SCALP} channels)"
        L.append(f"| {sid} | {pid} | {t1} | {len(kept)} | {len(raw_set)} ({len(canon_set)}) | {len(keep)} | "
                 f"{', '.join(c for c in kept if c not in canon_set) or '–'} | "
                 f"{', '.join(c for c in canon_set if c not in kept) or '–'} | {status} |")
        info[sid] = {"kept": kept, "set": raw_set, "used": keep}
        if len(keep) < MIN_SCALP:
            continue
        idx = [kept.index(c) for c in keep]
        st = d["norm_stats"]["eeg_orig"]["all"]
        z = d["eeg_orig"].float().numpy().astype(np.float64)
        v = z * np.asarray(st["mad"])[:, None] + np.asarray(st["med"])[:, None]          # back to microvolts
        v = v[idx]
        car = v - v.mean(0, keepdims=True)
        out = dict(d)
        out["eeg_names"] = keep
        out["norm_stats"] = dict(d["norm_stats"])
        for k, x in (("eeg_orig", v), ("eeg_car", car)):
            med, mad = robust(x)
            out[k] = torch.from_numpy(np.clip((x - med) / mad, -6e4, 6e4).astype(np.float16))
            out["norm_stats"][k] = {"all": {"med": med[:, 0].tolist(), "mad": mad[:, 0].tolist()}}
        out["channel_rule"] = "v1.6 kept ∩ authors' EEGLAB .set (Addendum A, post hoc)"
        torch.save(out, os.path.join(dst, f"{sid}.pt"))
        torch.save({k: out[k] for k in ("subject", "norm_stats", "norm_clip_z", "trial_bounds", "trial_session")},
                   os.path.join(dst, f"{sid}_normstats.pt"))
    md = "\n".join(L)
    print(md)
    open(os.path.join(a.out, "p17_setch.md"), "w", encoding="utf-8").write(md + "\n")
    json.dump(info, open(os.path.join(a.out, "p17_setch.json"), "w"), indent=1)


def report_setch(a):
    """H1 / H2 with the verified-electrode channel sets (post hoc), next to the registered primary result."""
    rng = np.random.default_rng(0)
    R = a.runs
    L = ["# Post-hoc channel sensitivity (Addendum A): H1 / H2 with verified scalp electrodes", "",
         "Not part of the pre-registered analysis; p-values unadjusted. The registered result (v1.6 channel rules) is "
         "shown for comparison.", "",
         "| analysis | comparison | n | mean A | mean B | median A − B | mean A − B [95 % CI] | A > B | p |",
         "|---|---|---|---|---|---|---|---|---|"]
    out = {}
    for tag, lab in (("primary", "registered (v1.6 channels)"), ("primary_setch", "post hoc (verified electrodes)")):
        F = _load(os.path.join(R, "main"), f"final_{tag}")
        L3 = _load(os.path.join(R, "no_env"), f"learned3_{tag}")
        F3 = _load(os.path.join(R, "no_env"), f"fixed3_{tag}")
        for name, A, B in (("H1: Final − Base", _ex(F, "combo_nosess"), _ex(F, "base")),
                           ("H2: Learned-3 − Fixed-3", _ex(L3, "combo_nosess"), _ex(F3, "combo_nosess"))):
            h = compare(A, B, rng)
            if h is None:
                continue
            out[f"{tag}|{name}"] = h
            L.append(f"| {lab} | {name} | {h['n']} | {h['mean_a']:.4f} | {h['mean_b']:.4f} | {h['median_diff']:+.4f} | "
                     f"{h['mean_diff']:+.4f} [{h['ci'][0]:+.4f}, {h['ci'][1]:+.4f}] | {h['wins']}/{h['n']} | {h['p']:.5f} |")
    for name in ("H1: Final − Base", "H2: Learned-3 − Fixed-3"):
        a_, b_ = out.get(f"primary|{name}"), out.get(f"primary_setch|{name}")
        if a_ and b_:
            L.append("")
            L.append(f"- {name}, per patient (registered → post hoc): " + ", ".join(
                f"{s} {a_['per_patient'][s]:+.4f} → {b_['per_patient'][s]:+.4f}" for s in b_["per_patient"]
                if s in a_["per_patient"]))
    md = "\n".join(L)
    print(md)
    open(os.path.join(R, "p17_setch_report.md"), "w", encoding="utf-8").write(md + "\n")
    json.dump(out, open(os.path.join(R, "p17_setch_report.json"), "w"), indent=1)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=("build", "runs", "report", "setch", "report_setch"))
    ap.add_argument("--sc_root", help="ScaleOLDNEW root of Phase 14 (proc/, raw/)")
    ap.add_argument("--out", help="build: output root (one folder per level)")
    ap.add_argument("--data", help="runs: primary data folder; report: build output root")
    ap.add_argument("--enc_root", help="runs: folder with <variant>/enc_final.pt")
    ap.add_argument("--runs", help="runs / report: Phase-17 run root")
    ap.add_argument("--variants", nargs="*", default=["main", "seed1", "seed2", "short", "no_env"])
    ap.add_argument("--patients", nargs="*", default=None)
    a = ap.parse_args()
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    {"build": build, "runs": runs, "report": report, "setch": setch, "report_setch": report_setch}[a.cmd](a)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/s2i/readout.py
"""
Phase 4 - readout improvements on the SAVED shared encoders (CPU, no retraining).

Phase 3 analysis of the shared encoder (15 subjects, 20 % calibration):
  (1) training traded high frequencies for low ones: the untrained Morlet bank (enc0) beats the
      trained encoder in alpha/beta, the trained one wins in delta/theta; two pairs collapsed on 50 Hz;
  (2) half of the features (pairs >= 13 Hz and their envelopes) sit in bands with little predictable
      signal, but one shared ridge penalty treats them like the useful ones;
  (3) at 1-2 min of calibration the readout (C x 40 weights per contact) is the bottleneck.
Every candidate below keeps the encoder fixed and changes only the readout:

  enc         reference = enc_ridge of s2i.shared (ridge, one K-fold alpha)
  base        reference = ridge_kfold_shared on LAG9
  enc_low     only low-frequency encoder features (pairs < 13 Hz, their envelopes, free filters)
  enc_group   high-frequency features down-weighted by a K-fold-chosen factor (two penalties)
  enc_enc0hi  + the UNTRAINED encoder's high-frequency features (alpha/beta), own K-fold weight
  enc_lag9    + LAG9 (raw lags +-200 ms), own K-fold weight
  enc_bands   target split into 5 complementary bands (<4, 4-8, 8-13, 13-30, >30 Hz); one ridge per
              band with its own K-fold penalty (chosen by MSE); predictions summed
  enc_sess    encoder features standardised per recording session (EEG only, label-free,
              transductive: uses the EEG of every trial of the session)
  enc_warp    filters time-scaled by s in {0.9, 1, 1.1} (frequencies x 1/s), s chosen per subject by
              K-fold (individual frequency adaptation)
  enc_bilin   structured readout W_k = a_k b_k^T (channels x features, rank 1-3, spectral profiles b
              orthonormal, ridge penalty on a), rank and penalty by K-fold: (C + F) R weights per contact
  ens_base    enc and base averaged with a K-fold-chosen weight
  ens_seed    enc of this run and of the seed-1 run averaged with a K-fold-chosen weight (if present)
--strict: undo preprocessing's whole-session normalisation and re-normalise EEG and target with the
calibration trials' statistics only; session-standardised models are skipped (combo_nosess = combo
without session standardisation is reported in every mode).
Phase 4b (--only_combo, combinations of the three Phase-4 winners):
  ens_all       ridge readouts of ALL available encoders (main + seed runs) averaged
  ens_all_sess  same, every encoder's features standardised per session
  combo         ens_all_sess mixed with base (LAG9 ridge), weight chosen by K-fold

Exactness: every readout is linear in its features, so K-fold out-of-fold r, the amplitude gain and
test r / R^2 are computed EXACTLY from second-order sufficient statistics (sums of x, xx', y, y^2,
xy') accumulated in one pass over the trials. `enc` therefore reproduces enc_ridge of s2i.shared.
Test trials, calibration fractions and fold blocks are those of phases 0-3.

Usage
-----
python -m s2i.readout --data DATA_DIR --run RUNS_DIR/shared_bip_car [--seed_run RUNS_DIR/shared_seed1]
python -m s2i.readout --data DATA_DIR --run RUNS_DIR/shared_bip_car --only_combo
       --seed_runs RUNS_DIR/shared_seed1 RUNS_DIR/shared_seed2     (Phase 4b: combinations of the winners)
"""
import argparse
import glob
import json
import os
import sys
import time

import numpy as np
import torch
from scipy.signal import butter, sosfiltfilt

from s2i.ceiling import ALPHAS, chrono_split, feat_lag
from s2i.baselines import DEFAULT_LAGS
from s2i.metrics import per_contact_metrics
from s2i.shared import TemporalEncoder, np_feat, r_single_trial, r_without_line
from s2i.within import ALPHAS_WIDE, _blocks

FS = 200
HIGH_HZ = 13.0
Y_CUTS = (4.0, 8.0, 13.0, 30.0)                       # -> 5 complementary target bands
GROUP_SCALES = (1.0, 0.5, 0.25, 0.1, 0.0)             # weight of the high-frequency group
ADD_SCALES = (0.0, 0.25, 0.5, 1.0, 2.0)               # weight of an added feature block
ENS_W = (0.0, 0.25, 0.5, 0.75, 1.0)                   # weight of enc in an ensemble
WARPS = (0.9, 1.1)
BILIN = tuple((r, al) for r in (1, 2, 3) for al in (0.001, 0.01, 0.1, 1.0, 10.0))
LAG9 = feat_lag(DEFAULT_LAGS)
_LP = {c: butter(4, c, btype="low", fs=FS, output="sos") for c in Y_CUTS}
_DEV = None                                           # set in main(); Gram matrices on the GPU if cuda


# ----------------------------------------------------------------------------- statistics
class St:
    """Second-order sufficient statistics of (features x, target y); optional band targets."""

    def __init__(self, D, K, nb=0, Db=0):
        self.n = 0.0
        self.sx, self.sxx = np.zeros(D), np.zeros((D, D))
        self.sy, self.syy, self.sxy = np.zeros(K), np.zeros(K), np.zeros((D, K))
        self.syb, self.syyb, self.sxyb = np.zeros((nb, K)), np.zeros((nb, K)), np.zeros((nb, Db, K))

    def add(self, X, y, yb=None):
        self.n += X.shape[1]
        self.sx += X.sum(1)
        if _DEV is not None and _DEV.type == "cuda":             # large Gram matrices on the GPU
            Xt, yt = torch.from_numpy(X).to(_DEV), torch.from_numpy(np.ascontiguousarray(y)).to(_DEV)
            self.sxx += (Xt @ Xt.T).double().cpu().numpy()
            self.sxy += (Xt @ yt.T).double().cpu().numpy()
        else:
            self.sxx += X @ X.T
            self.sxy += X @ y.T
        self.sy += y.sum(1)
        self.syy += (y * y).sum(1)
        if yb is not None:
            Xb = X[:self.sxyb.shape[1]]
            self.syb += yb.sum(2)
            self.syyb += (yb * yb).sum(2)
            for b in range(len(yb)):
                self.sxyb[b] += Xb @ yb[b].T

    def _new(self):
        return St.__new__(St)

    def __add__(self, o):
        s = self._new()
        for k in ("n", "sx", "sxx", "sy", "syy", "sxy", "syb", "syyb", "sxyb"):
            setattr(s, k, getattr(self, k) + getattr(o, k))
        return s

    def __sub__(self, o):
        s = self._new()
        for k in ("n", "sx", "sxx", "sy", "syy", "sxy", "syb", "syyb", "sxyb"):
            setattr(s, k, getattr(self, k) - getattr(o, k))
        return s

    def take(self, idx, band=None, transform=None):
        """Sub-block of features `idx` (band b replaces the target), optionally x -> (x - m) / d."""
        s = self._new()
        s.n, sx, sxx = self.n, self.sx[idx], self.sxx[np.ix_(idx, idx)]
        if band is None:
            s.sy, s.syy, sxy = self.sy, self.syy, self.sxy[idx]
        else:
            s.sy, s.syy, sxy = self.syb[band], self.syyb[band], self.sxyb[band][idx]
        if transform is not None:
            m, d = transform
            sxx = (sxx - np.outer(m, sx) - np.outer(sx, m) + s.n * np.outer(m, m)) / np.outer(d, d)
            sxy = (sxy - np.outer(m, s.sy)) / d[:, None]
            sx = (sx - s.n * m) / d
        s.sx, s.sxx, s.sxy = sx, sxx, sxy
        K = len(s.sy)
        s.syb, s.syyb, s.sxyb = np.zeros((0, K)), np.zeros((0, K)), np.zeros((0, len(idx), K))
        return s


def total(sts):
    out = sts[0]
    for s in sts[1:]:
        out = out + s
    return out


# ----------------------------------------------------------------------------- linear readouts
def standardise(tr):
    n = tr.n
    mu, ym = tr.sx / n, tr.sy / n
    G = tr.sxx - n * np.outer(mu, mu)
    sd = np.sqrt(np.clip(np.diag(G) / n, 0, None)) + 1e-8
    Gs = G / np.outer(sd, sd)
    Bs = (tr.sxy - n * np.outer(mu, ym)) / sd[:, None]
    return mu, sd, (Gs + Gs.T) / 2, Bs, ym


def to_raw(Ws, mu, sd, ym):
    A = Ws / sd[:, None]
    return A, ym - mu @ A


def ridge_models(tr, alphas, scale_vec=None):
    """Ridge for each alpha (penalty alpha * mean eigenvalue). scale_vec multiplies standardised features."""
    mu, sd, Gs, Bs, ym = standardise(tr)
    sc = max(np.trace(Gs) / len(Gs), 1e-12)      # penalty scale of the UNSCALED design (= n)
    if scale_vec is not None:
        Gs, Bs = Gs * np.outer(scale_vec, scale_vec), Bs * scale_vec[:, None]
    lam, Q = np.linalg.eigh(Gs)
    lam = np.clip(lam, 0, None)
    QB = Q.T @ Bs
    out = []
    for a in alphas:
        Ws = Q @ (QB / (lam + a * sc)[:, None])
        if scale_vec is not None:
            Ws = Ws * scale_vec[:, None]
        out.append(to_raw(Ws, mu, sd, ym))
    return out


def bilinear_models(tr, C, F, cands, iters=5):
    """
    W_k = a_k b_k^T over (channel, feature) with rank R: b_k (F x R) has orthonormal columns (spectral
    profiles), a_k (C x R) carries the ridge penalty alpha * n, exactly the ridge scale. Alternating
    least squares, initialised by the SVD of a ridge fit.
    """
    mu, sd, Gs, Bs, ym = standardise(tr)
    K, sc = Bs.shape[1], max(np.trace(Gs) / len(Gs), 1e-12)
    G4, B3 = Gs.reshape(C, F, C, F), Bs.reshape(C, F, K)
    lam, Q = np.linalg.eigh(Gs)
    W0 = Q @ ((Q.T @ Bs) / (np.clip(lam, 0, None) + sc)[:, None])
    _, _, Vt = np.linalg.svd(W0.T.reshape(K, C, F), full_matrices=False)
    out = []
    for R, alpha in cands:
        R_ = min(R, Vt.shape[1])
        b = Vt[:, :R_, :].transpose(0, 2, 1)                     # (K, F, R) orthonormal columns

        def solve_a(b):
            # Ga[k] = (I_C x b_k)' G (I_C x b_k): contract g with b (one matmul), then f (batched matmul)
            M = (Gs.reshape(C * F * C, F) @ b.transpose(1, 0, 2).reshape(F, K * R_)).reshape(C, F, C, K, R_)
            Ga = np.matmul(M.transpose(3, 0, 2, 4, 1), b[:, None, None]).transpose(0, 1, 4, 2, 3)
            ca = np.einsum("cfk,kfr->kcr", B3, b).reshape(K, C * R_, 1)
            return np.linalg.solve(Ga.reshape(K, C * R_, C * R_) + alpha * sc * np.eye(C * R_),
                                   ca)[..., 0].reshape(K, C, R_)
        a = solve_a(b)
        for _ in range(iters):
            # Gb[k] = (a_k x I_F)' G (a_k x I_F): contract c with a (one matmul), then d (batched matmul)
            M = (a.transpose(0, 2, 1).reshape(K * R_, C) @ Gs.reshape(C, F * C * F)).reshape(K, R_, F, C, F)
            Gb = np.matmul(M.transpose(0, 1, 2, 4, 3), a[:, None, None]).transpose(0, 2, 1, 3, 4)
            Gb = Gb.reshape(K, F * R_, F * R_)
            cb = np.einsum("cfk,kcr->kfr", B3, a).reshape(K, F * R_, 1)
            b = np.linalg.solve(Gb + 1e-4 * sc * np.eye(F * R_), cb)[..., 0].reshape(K, F, R_)
            b, _ = np.linalg.qr(b)                               # keep only the profile subspace
            a = solve_a(b)
        Ws = np.einsum("kcr,kfr->cfk", a, b).reshape(C * F, K)
        out.append(to_raw(Ws, mu, sd, ym))
    return out


def pred_sums(A, c, te):
    """Sums of prediction p = x'A + c over the samples summarised by `te`: (n, sp, spp, spy, sy, syy)."""
    Ax = A.T @ te.sx
    sp = Ax + te.n * c
    spp = ((te.sxx @ A) * A).sum(0) + 2 * c * Ax + te.n * c ** 2
    spy = (A * te.sxy).sum(0) + c * te.sy
    return np.stack([np.full_like(sp, te.n), sp, spp, spy, te.sy, te.syy])


def r_of(s):
    n, sp, spp, spy, sy, syy = s
    den = np.sqrt(np.clip(n * spp - sp ** 2, 0, None) * np.clip(n * syy - sy ** 2, 0, None))
    return np.where(den > 0, (n * spy - sp * sy) / np.maximum(den, 1e-12), np.nan)


def sse_of(s):
    return s[2] - 2 * s[3] + s[5]


class Fit:
    """K-fold selection over candidates from statistics. fit(train_stats) -> [(A, c)] per candidate."""

    def __init__(self, fit, folds, tot, crit="r"):
        self.fit, self.tot = fit, tot
        self.oof = None
        for tr, te in folds:
            s = np.stack([pred_sums(A, c, te) for A, c in fit(tr)])
            self.oof = s if self.oof is None else self.oof + s
        if crit == "r":
            score = [np.nanmean(r_of(o)) for o in self.oof]
            self.best = int(np.nanargmax(score))
        else:
            self.best = int(np.argmin([np.nansum(sse_of(o)) for o in self.oof]))
        self.A, self.c = fit(tot)[self.best]
        self.fold_models = None

    def fold_model(self, folds):
        if self.fold_models is None:
            self.fold_models = [self.fit(tr)[self.best] for tr, _ in folds]
        return self.fold_models


def calibrated(A, c, oof, ym):
    """Per-contact least-squares gain on the pooled out-of-fold predictions (r unchanged, R^2 honest)."""
    n, sp, spp, spy, sy, _ = oof
    num = spy - ym * sp - ym * sy + n * ym ** 2
    den = spp - 2 * ym * sp + n * ym ** 2
    g = np.clip(num / np.maximum(den, 1e-12), 0, None)
    return A * g, ym + g * (c - ym)


def score(A, c, oof, tot, te, te_sessions=None):
    ym = tot.sy / tot.n
    s = pred_sums(A, c, te)
    Ac, cc = calibrated(A, c, oof, ym)
    sc = pred_sums(Ac, cc, te)
    r2 = 1 - sse_of(sc) / np.maximum(sc[5] - sc[4] ** 2 / sc[0], 1e-12)
    out = {"r": float(np.nanmean(r_of(s))), "r2": float(np.nanmean(r2)), "contacts": [float(x) for x in r_of(s)],
           "cv_contacts": [float(x) for x in r_of(oof)]}          # K-fold r on calibration (reliability map)
    if te_sessions:                                            # calibration longevity: r per test session
        out["sessions"] = {k: float(np.nanmean(r_of(pred_sums(A, c, v)))) for k, v in te_sessions.items()}
    return out


def embed(A, c, idx, D):
    out = np.zeros((D, A.shape[1]))
    out[idx] = A
    return out, c


# ----------------------------------------------------------------------------- features
def load_encoder(path, dev):
    ck = torch.load(path, map_location=dev, weights_only=False)
    enc = TemporalEncoder(**ck["cfg"]).to(dev)
    enc.load_state_dict(ck["state"])
    return enc.eval()


def warped(enc, s, dev):
    """Same encoder with every kernel time-scaled by s: h_s(t) = h(s t), i.e. frequencies x 1/s."""
    out = TemporalEncoder(**enc.cfg).to(dev)
    w = enc.conv.weight.detach().cpu().numpy()[:, 0]
    H = w.shape[1] // 2
    t = np.arange(-H, H + 1)
    ws = np.stack([np.interp(t / s, t, k, left=0.0, right=0.0) for k in w])
    ws *= np.linalg.norm(w, axis=1, keepdims=True) / (np.linalg.norm(ws, axis=1, keepdims=True) + 1e-12)
    with torch.no_grad():
        out.conv.weight.copy_(torch.tensor(ws, dtype=torch.float32)[:, None])
    return out.eval()


def high_mask(enc, use_init):
    """Per-feature mask (length n_feat): True for quadrature pairs >= HIGH_HZ and their envelopes."""
    cfg = enc.cfg
    P = cfg["n_pairs"]
    if use_init:
        f = np.geomspace(cfg["fmin"], cfg["fmax"], P)
    else:
        w = enc.conv.weight.detach().cpu().numpy()[:, 0]
        spec = np.abs(np.fft.rfft(w, 4096, axis=1)) ** 2
        fr = np.fft.rfftfreq(4096, 1 / FS)
        f = fr[np.argmax(spec[0:2 * P:2] + spec[1:2 * P:2], 1)]
    m = np.zeros(enc.n_feat, bool)
    for p in np.where(f >= HIGH_HZ)[0]:
        m[[2 * p, 2 * p + 1]] = True
        if cfg["envelope"]:
            m[2 * P + cfg["n_free"] + p] = True
    return m, f


def band_targets(y):
    lp = [sosfiltfilt(_LP[c], y, axis=1) for c in Y_CUTS]
    parts = [lp[0]] + [lp[i] - lp[i - 1] for i in range(1, len(lp))] + [y - lp[-1]]
    return np.stack(parts).astype(np.float32)


# ----------------------------------------------------------------------------- per subject
def load_subject(path):
    """-> trials of CAR EEG, trials of bipolar targets, trial sessions, stimulus events (or None)."""
    d = torch.load(path, weights_only=False)
    eeg, tgt = d["eeg_car"].half().float().numpy(), d["ieeg_bip"].half().float().numpy()
    b = d["trial_bounds"].numpy()
    ses = list(d.get("trial_session", ["ses-01"] * len(b)))
    ev = d.get("stim_events")
    return ([eeg[:, s0:s1] for s0, s1 in b], [tgt[:, s0:s1] for s0, s1 in b], ses,
            None if ev is None else np.asarray(ev, int).reshape(-1, 2))


EPOCH = (-100, 500)                                   # -0.5 to +2.5 s around a stimulus marker (samples)


def r_event(P, T, test, events):
    """
    Per-contact r without the stimulus-locked component: each side minus ITS OWN average over the test stimulus
    epochs (-0.5 to +2.5 s around each stimulus marker). events: rows (trial index, onset sample within the trial).
    An epoch is used only if it lies inside one test trial / window; a sample covered by two epochs takes the
    later one. Returns NaN per contact if there is no such epoch.
    """
    pos = {int(w): j for j, w in enumerate(test)}
    ep = sorted((pos[int(w)], int(o)) for w, o in events
                if int(w) in pos and o + EPOCH[0] >= 0 and o + EPOCH[1] <= P[pos[int(w)]].shape[1])
    if not ep:
        return np.full(P[0].shape[0], np.nan)
    tP = np.mean([P[j][:, o + EPOCH[0]:o + EPOCH[1]] for j, o in ep], 0)
    tT = np.mean([T[j][:, o + EPOCH[0]:o + EPOCH[1]] for j, o in ep], 0)
    Pr, Tr = [p.astype(np.float64) for p in P], [t.astype(np.float64) for t in T]
    for j, o in ep:                                   # ascending: a later epoch overwrites the overlap
        Pr[j][:, o + EPOCH[0]:o + EPOCH[1]] = P[j][:, o + EPOCH[0]:o + EPOCH[1]] - tP
        Tr[j][:, o + EPOCH[0]:o + EPOCH[1]] = T[j][:, o + EPOCH[0]:o + EPOCH[1]] - tT
    return per_contact_metrics(Pr, Tr, FS, bands=False)["r"]


def _robust(x):
    med = np.median(x, axis=1, keepdims=True)
    mad = np.median(np.abs(x - med), axis=1, keepdims=True) * 1.4826
    return med, np.where(mad < 1e-12, 1.0, mad)


def strict_renorm(E, Y, ses, stats, n_cal):
    """
    Undo the whole-session robust normalisation of preprocessing (x * MAD_s + median_s) and re-normalise every
    trial with the median / MAD of the CALIBRATION trials only (first n_cal), for EEG and target alike:
    no statistic of a test trial is used anywhere (strict, deployment-faithful normalisation).
    """
    def redo(X, key):
        raw = []
        for x, s in zip(X, ses):
            st = stats["norm_stats"][key][s]
            raw.append(x * np.asarray(st["mad"], np.float32)[:, None] + np.asarray(st["med"], np.float32)[:, None])
        med, mad = _robust(np.concatenate(raw[:n_cal], 1))
        return [((r - med) / mad).astype(np.float32) for r in raw]
    return redo(E, "eeg_car"), redo(Y, "ieeg_bip")


def analyze(E, Y, ses, enc, seed_encs, a, dev, log, events=None):
    amp = dev.type == "cuda"
    zoo = not a.only_combo
    n, C, K = len(E), E[0].shape[0], Y[0].shape[0]
    _, test = chrono_split(n, 0.2, gap=1)
    enc0 = TemporalEncoder(**enc.cfg).to(dev).eval()
    F = enc.n_feat
    hi_enc, f_enc = high_mask(enc, use_init=False)
    hi_e0, _ = high_mask(enc0, use_init=True)
    f1, f0 = np_feat(enc, dev, amp), np_feat(enc0, dev, amp)
    fss = [np_feat(se, dev, amp) for se in seed_encs]
    fs1 = fss[0] if fss else None
    fw = {s: np_feat(warped(enc, s, dev), dev, amp) for s in WARPS} if zoo else {}
    e0_rows = np.where(np.tile(hi_e0, C))[0]
    seeds = [f"seed{j}" for j in range(len(fss))]

    # main view = [enc | enc0 high-frequency rows (zoo only) | LAG9 | seed encoders]
    blocks = [("enc", C * F)] + ([("enc0hi", len(e0_rows))] if zoo else []) + [("lag9", C * len(DEFAULT_LAGS))]
    blocks += [(b, C * se.n_feat) for b, se in zip(seeds, seed_encs)]     # encoders may differ (e.g. no_env)
    idx, o = {}, 0
    for name, size in blocks:
        idx[name] = np.arange(o, o + size)
        o += size
    D = o
    hi_rows = np.where(np.tile(hi_enc, C))[0]
    lo_rows = np.where(~np.tile(hi_enc, C))[0]

    def main_x(e):
        x1 = f1(e)
        parts = [x1] + ([f0(e)[e0_rows]] if zoo else []) + [LAG9(e)] + [f(e) for f in fss]
        return np.concatenate(parts).astype(np.float32), x1

    # session statistics of every encoder's features (EEG of every trial, no target)
    enc_fns = {"enc": f1, **dict(zip(seeds, fss))}
    sess_sum = {}
    for i in range(n):
        for b, fn in enc_fns.items():
            x = fn(E[i]).astype(np.float64)
            s = sess_sum.setdefault((b, ses[i]), [0, np.zeros(len(idx[b])), np.zeros(len(idx[b]))])
            s[0] += x.shape[1]
            s[1] += x.sum(1)
            s[2] += (x * x).sum(1)
    sess_tf = {k: (v[1] / v[0], np.sqrt(np.clip(v[2] / v[0] - (v[1] / v[0]) ** 2, 0, None)) + 1e-6)
               for k, v in sess_sum.items()}

    def tf_for(names, session):
        """Per-session standardisation of encoder blocks; identity for other blocks (e.g. LAG9)."""
        m = [sess_tf[(b, session)][0] if (b, session) in sess_tf else np.zeros(len(idx[b])) for b in names]
        d = [sess_tf[(b, session)][1] if (b, session) in sess_tf else np.ones(len(idx[b])) for b in names]
        return np.concatenate(m), np.concatenate(d)

    # test statistics (per session, for the session-standardised readout)
    te_main, te_w = {}, {s: St(C * F, K) for s in fw}
    for i in test:
        X, _ = main_x(E[i])
        te_main.setdefault(ses[i], St(D, K)).add(X, Y[i])
        for s in fw:
            te_w[s].add(fw[s](E[i]).astype(np.float32), Y[i])
    te = total(list(te_main.values()))
    te_enc = te.take(idx["enc"])
    te_sess = total([v.take(idx["enc"], transform=tf_for(["enc"], k)) for k, v in te_main.items()])

    order = list(dict.fromkeys(ses))                     # sessions in recording order
    res = {"n_trials": n, "n_test": len(test), "n_sessions": len({k[1] for k in sess_sum}),
           "session_order": order, "test_trials_per_session": {k: sum(ses[i] == k for i in test) for k in order},
           "n_encoders": 1 + len(fss),
           "pair_peak_hz": [float(x) for x in f_enc], "fracs": {}}
    for frac in a.fracs:
        n_c = max(3, int(round(n * frac)))
        if n_c >= test[0]:
            continue
        t0 = time.time()
        fold_ids = _blocks(n_c, 5)
        cal = {}                                 # (fold, session) -> St (main view, with band targets)
        cal_w = {s: [St(C * F, K) for _ in fold_ids] for s in fw}
        nbt = len(Y_CUTS) + 1 if zoo else 0
        for fi, ids in enumerate(fold_ids):
            for i in ids:
                X, _ = main_x(E[i])
                cal.setdefault((fi, ses[i]), St(D, K, nb=nbt, Db=C * F if zoo else 0)).add(
                    X, Y[i], band_targets(Y[i]) if zoo else None)
                for s in fw:
                    cal_w[s][fi].add(fw[s](E[i]).astype(np.float32), Y[i])
        fold_st = [total([v for (f, _), v in cal.items() if f == fi]) for fi in range(len(fold_ids))]
        tot = total(fold_st)

        def folds_of(take):
            return [(take(tot - fs), take(fs)) for fs in fold_st]

        out, extra = {}, {}

        def run(name, take, fit, test_st, crit="r", note=None):
            fl = folds_of(take)
            m = Fit(fit, fl, take(tot), crit)
            out[name] = score(m.A, m.c, m.oof[m.best], take(tot), test_st,
                              {k: take(v) for k, v in te_main.items()})
            if note:
                extra[name] = note(m)
            return m, fl

        t_enc = lambda s: s.take(idx["enc"])                                    # noqa: E731
        m_enc, fl_enc = run("enc", t_enc, lambda tr: ridge_models(tr, ALPHAS_WIDE), te_enc,
                            note=lambda m: ALPHAS_WIDE[m.best])
        t_lag = lambda s: s.take(idx["lag9"])                                   # noqa: E731
        m_base, fl_base = run("base", t_lag, lambda tr: ridge_models(tr, ALPHAS), te.take(idx["lag9"]))
        store = {}                       # model -> (feature rows, calibrated A, c, session-standardised blocks)
        for nm, m_, rows in (("enc", m_enc, idx["enc"]), ("base", m_base, idx["lag9"])):
            ts = tot.take(rows)
            store[nm] = (rows, *calibrated(m_.A, m_.c, m_.oof[m_.best], ts.sy / ts.n), None)

        if zoo:
            lo = idx["enc"][lo_rows]
            run("enc_low", lambda s: s.take(lo), lambda tr: ridge_models(tr, ALPHAS_WIDE), te.take(lo))

            gmask = np.zeros(C * F, bool)
            gmask[hi_rows] = True
            cands = [(g, al) for g in GROUP_SCALES for al in ALPHAS_WIDE]

            def fit_group(tr):
                outm = []
                for g in GROUP_SCALES:
                    outm += ridge_models(tr, ALPHAS_WIDE, np.where(gmask, g, 1.0))
                return outm
            run("enc_group", t_enc, fit_group, te_enc, note=lambda m: cands[m.best][0])

            def added(name, block):
                sel = np.concatenate([idx["enc"], idx[block]])
                mask = np.r_[np.zeros(len(idx["enc"]), bool), np.ones(len(idx[block]), bool)]
                cands_ = [(g, al) for g in ADD_SCALES for al in ALPHAS_WIDE]

                def fit(tr):
                    outm = []
                    for g in ADD_SCALES:
                        outm += ridge_models(tr, ALPHAS_WIDE, np.where(mask, g, 1.0))
                    return outm
                run(name, lambda s: s.take(sel), fit, te.take(sel), note=lambda m: cands_[m.best][0])
            added("enc_enc0hi", "enc0hi")
            added("enc_lag9", "lag9")

            # one ridge per target band (penalty by MSE on that band), predictions summed
            nb = len(Y_CUTS) + 1
            bm = [Fit(lambda tr: ridge_models(tr, ALPHAS_WIDE), folds_of(lambda s, b=b: s.take(idx["enc"], band=b)),
                      tot.take(idx["enc"], band=b), crit="mse") for b in range(nb)]
            A = sum(m.A for m in bm)
            c = sum(m.c for m in bm)
            fms = [m.fold_model([(tr, None) for tr, _ in folds_of(lambda s, b=b: s.take(idx["enc"], band=b))])
                   for b, m in enumerate(bm)]
            oof = sum(pred_sums(sum(fms[b][fi][0] for b in range(nb)), sum(fms[b][fi][1] for b in range(nb)), te_f)
                      for fi, (_, te_f) in enumerate(fl_enc))
            out["enc_bands"] = score(A, c, oof, t_enc(tot), te_enc)
            extra["enc_bands"] = [ALPHAS_WIDE[m.best] for m in bm]

            # per-session standardisation of the encoder features
            sess_tot = total([v.take(idx["enc"], transform=tf_for(["enc"], s)) for (f, s), v in cal.items()])
            sess_folds = [total([v.take(idx["enc"], transform=tf_for(["enc"], s)) for (f, s), v in cal.items()
                                 if f == fi])
                          for fi in range(len(fold_ids))]
            m = Fit(lambda tr: ridge_models(tr, ALPHAS_WIDE), [(sess_tot - fs, fs) for fs in sess_folds], sess_tot)
            out["enc_sess"] = score(m.A, m.c, m.oof[m.best], sess_tot, te_sess)

            # individual frequency adaptation: pick s in {0.9, 1, 1.1} by K-fold r
            best = ("1.0", m_enc, np.nanmean(r_of(m_enc.oof[m_enc.best])), te_enc, t_enc(tot))
            for s in WARPS:
                tw = total(cal_w[s])
                mw = Fit(lambda tr: ridge_models(tr, ALPHAS_WIDE), [(tw - fs, fs) for fs in cal_w[s]], tw)
                sw = np.nanmean(r_of(mw.oof[mw.best]))
                if sw > best[2]:
                    best = (f"{s:g}", mw, sw, te_w[s], tw)
            out["enc_warp"] = score(best[1].A, best[1].c, best[1].oof[best[1].best], best[4], best[3])
            extra["enc_warp"] = best[0]

            # structured (bilinear) readout
            if not a.no_bilin:
                run("enc_bilin", t_enc, lambda tr: bilinear_models(tr, C, F, BILIN), te_enc,
                    note=lambda m: list(BILIN[m.best]))

        # ensembles: enc + base, enc + seed-1 (weights chosen by pooled out-of-fold r)
        def ensemble(name, m1, fl1, i1, m2, fl2, i2):
            sel = np.concatenate([i1, i2])
            n1 = len(i1)
            fm1 = m1.fold_model(fl1)
            fm2 = m2.fold_model(fl2)
            t_sel = lambda s: s.take(sel)                                       # noqa: E731
            fl = folds_of(t_sel)
            oofs = []
            for w in ENS_W:
                oofs.append(sum(pred_sums(np.r_[w * fm1[fi][0], (1 - w) * fm2[fi][0]], w * fm1[fi][1] +
                                          (1 - w) * fm2[fi][1], te_f) for fi, (_, te_f) in enumerate(fl)))
            wi = int(np.nanargmax([np.nanmean(r_of(o)) for o in oofs]))
            w = ENS_W[wi]
            A_ = np.r_[w * m1.A, (1 - w) * m2.A]
            out[name] = score(A_, w * m1.c + (1 - w) * m2.c, oofs[wi], t_sel(tot), te.take(sel))
            extra[name] = w
            assert len(A_) == len(sel) and n1 == len(m1.A)
        ensemble("ens_base", m_enc, fl_enc, idx["enc"], m_base, fl_base, idx["lag9"])
        if fss:
            t_seed = lambda s: s.take(idx["seed0"])                             # noqa: E731
            m_seed, fl_seed = run("seed", t_seed, lambda tr: ridge_models(tr, ALPHAS_WIDE), te.take(idx["seed0"]))
            ensemble("ens_seed", m_enc, fl_enc, idx["enc"], m_seed, fl_seed, idx["seed0"])

        # combinations of the three Phase-4 winners: several encoders averaged (ens_seed), session
        # standardisation (enc_sess) and mixing with LAG9 ridge (ens_base); exact from statistics
        def mixture(name, encs, sess, fixed_w=None):
            names = encs + ["lag9"]
            sel = np.concatenate([idx[b] for b in names])

            def T(st, session):
                return st.take(sel, transform=tf_for(names, session) if sess else None)
            folds_T = [total([T(v, ss) for (f, ss), v in cal.items() if f == fi]) for fi in range(len(fold_ids))]
            tot_T = total(folds_T)
            te_T = total([T(v, ss) for ss, v in te_main.items()])
            comps, o_ = [], 0
            for b in names:
                loc = np.arange(o_, o_ + len(idx[b]))
                o_ += len(idx[b])
                al = ALPHAS if b == "lag9" else ALPHAS_WIDE
                fl = [((tot_T - fs).take(loc), fs.take(loc)) for fs in folds_T]
                m = Fit(lambda tr, al=al: ridge_models(tr, al), fl, tot_T.take(loc))
                comps.append((b, loc, m, m.fold_model(fl)))

            def combine(w, models):
                A_, c_ = np.zeros((len(sel), K)), np.zeros(K)
                for (b, loc, _, _), (Ab, cb) in zip(comps, models):
                    wt = (1 - w) if b == "lag9" else w / len(encs)
                    A_[loc] += wt * Ab
                    c_ += wt * cb
                return A_, c_
            ws = ENS_W if fixed_w is None else (fixed_w,)
            oofs = [sum(pred_sums(*combine(w, [cm[3][fi] for cm in comps]), fs) for fi, fs in enumerate(folds_T))
                    for w in ws]
            wi = int(np.nanargmax([np.nanmean(r_of(o)) for o in oofs]))
            A_, c_ = combine(ws[wi], [(cm[2].A, cm[2].c) for cm in comps])
            out[name] = score(A_, c_, oofs[wi], tot_T, te_T, {ss: T(v, ss) for ss, v in te_main.items()})
            store[name] = (sel, *calibrated(A_, c_, oofs[wi], tot_T.sy / tot_T.n), names if sess else None)
            extra[name] = ws[wi]

        all_encs = ["enc"] + seeds
        if not zoo and not a.strict:
            mixture("enc_sess", ["enc"], sess=True, fixed_w=1.0)
        if seeds:
            mixture("ens_all", all_encs, sess=False, fixed_w=1.0)
            if not a.strict:
                mixture("ens_all_sess", all_encs, sess=True, fixed_w=1.0)
        if not a.strict:                         # session standardisation uses the EEG of every trial (transductive)
            mixture("combo", all_encs, sess=True)
        mixture("combo_nosess", all_encs, sess=False)

        if a.explicit and abs(frac - a.explicit_frac) < 1e-9:
            # explicit test predictions of the final models: band / 50 Hz / single-trial metrics
            res["explicit"] = {}
            Yt = [Y[i] for i in test]
            for nm in [m for m in ("base", "enc", "combo", "combo_nosess") if m in store]:
                rows, A_, c_, names = store[nm]
                P = []
                for i in test:
                    X = main_x(E[i])[0][rows].astype(np.float64)
                    if names:
                        mm, dd = tf_for(names, ses[i])
                        X = (X - mm[:, None]) / dd[:, None]
                    P.append((X.T @ A_ + c_).T.astype(np.float32))
                met = per_contact_metrics(P, Yt, FS, bands=True)
                met["r_no50"], met["r_induced"] = r_without_line(P, Yt), r_single_trial(P, Yt)
                if events is not None:
                    met["r_event"] = r_event(P, Yt, test, events)
                Pc = np.concatenate(P, 1)
                res["explicit"][nm] = {"frac": frac, "mean": {k: float(np.nanmean(v)) for k, v in met.items()},
                                       "r": [float(x) for x in met["r"]],
                                       "r_induced": [float(x) for x in met["r_induced"]],
                                       "n_const": int((Pc.max(1) == Pc.min(1)).sum())}   # constant -> r = 0
            log("    explicit: " + " ".join(f"{k} r={v['mean']['r']:.4f} (stats {out[k]['r']:.4f}) "
                                          f"r_ind={v['mean']['r_induced']:.4f}"
                                          for k, v in res["explicit"].items()))
        mins = sum(E[i].shape[1] for i in range(n_c)) / FS / 60
        res["fracs"][f"{frac:g}"] = {"n_calib": n_c, "minutes": mins, "models": out, "choice": extra,
                                     "calib_sessions": sorted({ses[i] for i in range(n_c)}, key=order.index)}
        log(f"    f={frac:g} ({mins:.1f} min, {time.time() - t0:.0f}s) " +
            " ".join(f"{k}={v['r']:.3f}" for k, v in out.items()))
    return res


# ----------------------------------------------------------------------------- summary
def summarize(results, out_dir, log):
    from scipy.stats import wilcoxon
    lines, summ = [], {}
    fracs = sorted({f for r in results for f in r["fracs"]}, key=float)
    for f in fracs:
        rows = [r["fracs"][f] for r in results if f in r["fracs"]]
        models = [m for m in rows[0]["models"] if all(m in r["models"] for r in rows)]
        M = {m: np.array([r["models"][m]["r"] for r in rows]) for m in models}
        R2 = {m: np.array([r["models"][m]["r2"] for r in rows]) for m in models}
        mins = np.mean([r["minutes"] for r in rows])
        lines += [f"\n### calib fraction {f}  ({len(rows)} subjects, mean {mins:.1f} min)",
                  "| model | mean r | vs enc | wins vs enc | p vs enc | vs base | R² (calibrated) | K-fold choices |",
                  "|---|---|---|---|---|---|---|---|"]
        summ[f] = {}
        for m in models:
            d = M[m] - M["enc"]
            try:
                p = float(wilcoxon(M[m], M["enc"]).pvalue) if m != "enc" and np.any(d != 0) else float("nan")
            except ValueError:
                p = float("nan")
            ch = [r["choice"].get(m) for r in rows if m in r["choice"]]
            ch = [tuple(x) if isinstance(x, list) else x for x in ch]
            if ch and m != "enc_bands":
                vals, cnt = np.unique([str(x) for x in ch], return_counts=True)
                chs = ", ".join(f"{v}×{c}" for v, c in zip(vals, cnt))
            else:
                chs = ""
            summ[f][m] = {"mean": float(M[m].mean()), "diff_enc": float(d.mean()), "wins_enc": int((d > 0).sum()),
                          "p_enc": p, "diff_base": float((M[m] - M["base"]).mean()), "r2": float(R2[m].mean())}
            lines.append(f"| {m} | {M[m].mean():.4f} | {d.mean():+.4f} | {(d > 0).sum()}/{len(d)} | {p:.4f} | "
                         f"{(M[m] - M['base']).mean():+.4f} | {R2[m].mean():.3f} | {chs} |")
    md = "\n".join(lines)
    log(md)
    with open(os.path.join(out_dir, "readout.md"), "w", encoding="utf-8") as fh:
        fh.write(md + "\n")
    with open(os.path.join(out_dir, "readout.json"), "w") as fh:
        json.dump(summ, fh, indent=1)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--run", required=True, help="output dir of s2i.shared (enc_<sub>.pt)")
    ap.add_argument("--seed_run", default=None, help="second run for the seed ensemble (enc_<sub>.pt)")
    ap.add_argument("--seed_runs", nargs="*", default=[], help="further runs (encoders) for the ensemble")
    ap.add_argument("--explicit", action="store_true",
                    help="also predict the test trials explicitly with base / enc / combo (all metrics)")
    ap.add_argument("--explicit_frac", type=float, default=0.2)
    ap.add_argument("--strict", action="store_true",
                    help="re-normalise EEG and target with calibration-trial statistics only (needs <sub>_normstats.pt "
                         "from preprocess --stats_only); no session standardisation; requires a single --fracs value")
    ap.add_argument("--stats_dir", default=None, help="folder with <sub>_normstats.pt (default: --data)")
    ap.add_argument("--untrained", action="store_true",
                    help="replace every encoder by its Morlet initialisation (fixed filter bank, no meta-learning)")
    ap.add_argument("--only_combo", action="store_true",
                    help="skip the Phase-4 variants already rejected; compute enc, base, ensembles and combos")
    ap.add_argument("--out_name", default=None, help="output sub-folder (default: readout / readout_combo)")
    ap.add_argument("--fracs", type=float, nargs="*", default=[0.05, 0.1, 0.2])
    ap.add_argument("--subjects", nargs="*", default=None)
    ap.add_argument("--no_bilin", action="store_true")
    ap.add_argument("--overwrite", action="store_true")
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    a = ap.parse_args()
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    if a.strict:
        assert len(a.fracs) == 1, "--strict needs exactly one calibration fraction (the statistics depend on it)"
    out = os.path.join(a.run, a.out_name or ("readout_strict" if a.strict else
                                             "readout_combo" if a.only_combo else "readout"))
    os.makedirs(out, exist_ok=True)
    log_f = open(os.path.join(out, "readout.log"), "a", encoding="utf-8")

    def log(msg):
        line = f"{time.strftime('%H:%M:%S')} | {msg}"
        print(line, flush=True)
        log_f.write(line + "\n")
        log_f.flush()

    dev = torch.device(a.device)
    torch.set_grad_enabled(False)
    global _DEV
    _DEV = dev
    seed_runs = ([a.seed_run] if a.seed_run else []) + list(a.seed_runs)
    log(f"config: {vars(a)}")
    results = []
    for ep in sorted(glob.glob(os.path.join(a.run, "enc_sub-*.pt"))):
        sid = os.path.basename(ep)[4:-3]
        if a.subjects and sid not in a.subjects:
            continue
        fp = os.path.join(out, f"readout_{sid}.json")
        sps = [sp for sp in (os.path.join(r_, f"enc_{sid}.pt") for r_ in seed_runs) if os.path.exists(sp)]
        if not os.path.exists(os.path.join(a.data, f"{sid}.pt")):
            log(f"[{sid}] no data file in {a.data} (excluded at this level), skip")
            continue
        if os.path.exists(fp) and not a.overwrite:
            done = json.load(open(fp))
            # resume: keep a finished subject only if it used the same number of encoders as now
            if done.get("n_encoders", 1 + len(sps)) == 1 + len(sps):
                results.append(done)
                log(f"[{sid}] already done, loaded")
                continue
            log(f"[{sid}] done with {done['n_encoders']} encoders, now {1 + len(sps)}: recomputing")
        t0 = time.time()
        enc = load_encoder(ep, dev)
        seed_encs = [load_encoder(sp, dev) for sp in sps]
        if a.untrained:
            enc = TemporalEncoder(**enc.cfg).to(dev).eval()
            seed_encs = [TemporalEncoder(**se.cfg).to(dev).eval() for se in seed_encs]
        E, Y, ses, events = load_subject(os.path.join(a.data, f"{sid}.pt"))
        if a.strict:
            st = torch.load(os.path.join(a.stats_dir or a.data, f"{sid}_normstats.pt"), weights_only=False)
            assert len(st["trial_bounds"]) == len(E) and list(st["trial_session"]) == ses, "stats / data mismatch"
            E, Y = strict_renorm(E, Y, ses, st, max(3, int(round(len(E) * a.fracs[0]))))
        log(f"[{sid}] {len(E)} trials, {E[0].shape[0]} scalp, {Y[0].shape[0]} targets, "
            f"{len(set(ses))} sessions, extra encoders: {len(seed_encs)}")
        r = analyze(E, Y, ses, enc, seed_encs, a, dev, log, events)
        r["subject"] = sid
        with open(fp + ".tmp", "w") as fh:
            json.dump(r, fh)
        os.replace(fp + ".tmp", fp)
        results.append(r)
        log(f"[{sid}] done in {(time.time() - t0) / 60:.1f} min")
    if results:
        summarize(results, out, log)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/s2i/resample_check.py
"""
Timing of the executed resampling for every ds004752 recording (read-only check for the manuscript, Section 2.1).

MNE's FFT resampling (Raw.resample, default npad="auto") does not keep the nominal time axis exactly: the output is
shifted by a few milliseconds, by an amount that depends on the exact number of input samples, and the shift drifts
slowly along the record. Scalp EEG and SEEG of ds004752 are separate EDF files (scalp EEG of patients 1-9 already at
200 Hz, not resampled), so the two modalities can be offset relative to each other by a session-specific amount.

For every session this script
  1. reads only the EDF headers from OpenNeuro (HTTP range requests, a few kB per file): sampling rate and the exact
     number of samples (records x samples per record, as read by MNE);
  2. measures, for that exact (rate, length), the delay of a 10 Hz sinusoid after Raw.resample(200) at 10 %, 50 % and
     90 % of the record (filtering is zero-phase and adds no delay; resampling alone is measured);
  3. reports the SEEG - scalp offset at the three positions.
Negative delay = the output leads the nominal time axis.

Usage
-----
python -m s2i.resample_check --out OUT.md [--subjects sub-01 ...]
"""
import argparse
import os
import re
import sys
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET

import numpy as np

BUCKET = "https://s3.amazonaws.com/openneuro.org"
PREFIX = "ds004752/"
FS = 200
POS = (0.10, 0.50, 0.90)


def list_edf():
    keys, token = [], None
    while True:
        url = f"{BUCKET}?list-type=2&prefix={PREFIX}" + (f"&continuation-token={urllib.parse.quote(token)}" if token else "")
        root = ET.fromstring(urllib.request.urlopen(url, timeout=120).read())
        ns = {"s": root.tag.split("}")[0].strip("{")}
        for c in root.findall("s:Contents", ns):
            k = c.find("s:Key", ns).text
            if k.endswith(".edf") and "/derivatives/" not in k:
                keys.append((k, int(c.find("s:Size", ns).text)))
        if root.find("s:IsTruncated", ns).text != "true":
            return keys
        token = root.find("s:NextContinuationToken", ns).text


def get_range(key, a, b):
    req = urllib.request.Request(f"{BUCKET}/{key}", headers={"Range": f"bytes={a}-{b}"})
    return urllib.request.urlopen(req, timeout=120).read()


def edf_info(key, size):
    h = get_range(key, 0, 255)
    ns = int(h[252:256].decode("ascii").strip())
    full = get_range(key, 0, 256 * (ns + 1) - 1)
    n_rec = int(full[236:244].decode("ascii").strip())
    dur = float(full[244:252].decode("ascii").strip())
    off = 256 + ns * 216
    spr = [int(full[off + 8 * i:off + 8 * (i + 1)].decode("ascii").strip()) for i in range(ns)]
    if n_rec < 0:                                       # unknown record count: from the file size (int16 samples)
        n_rec = (size - 256 * (ns + 1)) // (2 * sum(spr))
    smax = max(spr)
    return smax / dur, n_rec * smax, sorted(set(spr))


_cache = {}


def delays(sf, n):
    """Delay (ms) of a 10 Hz sinusoid after Raw.resample(200) at the positions POS; 0 if no resampling."""
    if abs(sf - FS) < 1e-9:
        return (0.0,) * len(POS)
    key = (sf, n)
    if key not in _cache:
        import mne
        mne.set_log_level("ERROR")
        t = np.arange(n) / sf
        z = mne.io.RawArray(np.sin(2 * np.pi * 10 * t)[None], mne.create_info(["s"], sf, "eeg")).resample(FS).get_data()[0]
        to = np.arange(len(z)) / FS
        out = []
        for p in POS:
            sl = slice(int((p - 0.02) * len(z)), int((p + 0.02) * len(z)))
            ph = np.arctan2(z[sl] @ np.cos(2 * np.pi * 10 * to[sl]), z[sl] @ np.sin(2 * np.pi * 10 * to[sl]))
            out.append(float(-1e3 * ph / (2 * np.pi * 10)))
        _cache[key] = tuple(out)
    return _cache[key]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", required=True)
    ap.add_argument("--subjects", nargs="*", default=None)
    a = ap.parse_args()
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    sess = {}
    for key, size in list_edf():
        m = re.search(r"(sub-\d+)/(ses-\d+)/(eeg|ieeg)/", key)
        if not m or (a.subjects and m.group(1) not in a.subjects):
            continue
        sess.setdefault((m.group(1), m.group(2)), {})[m.group(3)] = (key, size)
    L = ["# Timing of the executed resampling, per ds004752 session",
         "", "Delay of a 10 Hz sinusoid after Raw.resample(200) for the exact rate and length of each EDF file, at 10 / 50 / "
         "90 % of the record (ms; negative = leads the nominal time axis). Offset = SEEG delay − scalp delay.", "",
         "| patient | session | scalp Hz | scalp samples | scalp delay | SEEG Hz | SEEG samples | SEEG delay | SEEG − scalp |",
         "|---|---|---|---|---|---|---|---|---|"]
    offs = []
    for (sub, ses), mods in sorted(sess.items()):
        if "eeg" not in mods or "ieeg" not in mods:
            continue
        fe, ne, _ = edf_info(*mods["eeg"])
        fi, ni, _ = edf_info(*mods["ieeg"])
        de, di = delays(fe, ne), delays(fi, ni)
        off = [x - y for x, y in zip(di, de)]
        offs.append((sub, off))
        f3 = lambda v: " / ".join(f"{x:+.2f}" for x in v)                       # noqa: E731
        L.append(f"| {sub} | {ses} | {fe:g} | {ne} | {f3(de)} | {fi:g} | {ni} | {f3(di)} | {f3(off)} |")
        print(L[-1], flush=True)
    allv = np.array([x for _, o in offs for x in o])
    if len(allv):
        L += ["", f"- SEEG − scalp offset over all sessions and positions: {allv.min():+.2f} to {allv.max():+.2f} ms "
              f"(median |offset| {np.median(np.abs(allv)):.2f} ms, {len(offs)} sessions)"]
        for sub in sorted({s for s, _ in offs}):
            v = np.array([x for s, o in offs if s == sub for x in o])
            L.append(f"  - {sub}: {v.min():+.2f} to {v.max():+.2f} ms")
    out = "\n".join(L)
    print(out)
    os.makedirs(os.path.dirname(os.path.abspath(a.out)), exist_ok=True)
    open(a.out, "w", encoding="utf-8").write(out + "\n")


if __name__ == "__main__":
    main()

In [ ]:
#@title 2b. Kiểm tra (cả ba số phải > 0)
!grep -c "def setch" {CODE_DIR}/s2i/scales17.py
!grep -c "def report_setch" {CODE_DIR}/s2i/scales17.py
!grep -c "def r_event" {CODE_DIR}/s2i/readout.py

## 19A. So sánh kênh và dựng dữ liệu `primary_setch`

In [ ]:
#@title 19A
!python -m s2i.scales17 setch --sc_root {SC_ROOT} --out {DATA17}

## 19B. Readout trên các kênh đã xác nhận (GPU)

In [ ]:
#@title 19B
D, R = f'{DATA17}/primary_setch', RUNS17
!python -m s2i.readout --data {D} --run {R}/main --seed_runs {R}/seed1 {R}/seed2 {R}/short {R}/no_env --only_combo --strict --fracs 0.2 --explicit --out_name final_primary_setch
!python -m s2i.readout --data {D} --run {R}/no_env --seed_runs {R}/short {R}/main --only_combo --strict --fracs 0.2 --explicit --out_name learned3_primary_setch
!python -m s2i.readout --data {D} --run {R}/no_env --seed_runs {R}/short {R}/main --untrained --only_combo --strict --fracs 0.2 --explicit --out_name fixed3_primary_setch

## 19C. Báo cáo

In [ ]:
#@title 19C
!python -m s2i.scales17 report_setch --runs {RUNS17}

## 19D (tuỳ chọn). Độ lệch thời gian do resample, từng session ds004752

Đọc header EDF từ OpenNeuro (vài KB mỗi file) để lấy số mẫu chính xác, rồi đo độ trễ sau `Raw.resample(200)` ở 10%, 50% và 90% bản ghi. Mình đã chạy bước này trên máy local; cell này để bạn tự tái lập.

In [ ]:
#@title 19D
!python -m s2i.resample_check --out {ROOT}/runs/checks18/resample_sessions.md

## Cách đọc kết quả
- **Bảng 19A:** với mỗi bệnh nhân, so sánh kênh đã giữ theo v1.6, kênh trong file `.set` của tác giả, và phần giao. Cột "kept, not in .set" là các kênh nghi vấn.
- **Bảng 19C:** H1/H2 trên kênh đã xác nhận, đặt cạnh kết quả đã đăng ký.
  - Hai kết quả gần nhau: tuyên bố tổng quát hoá không phụ thuộc vào các kênh nghi vấn.
  - Hai kết quả khác nhau: bài báo cáo cả hai; kết quả đã đăng ký vẫn là kết quả chính.

Gửi lại: toàn bộ output 19A và 19C.